     RESSET数据库缺少科创板的数据

# 中证800历史可交易股票池

本 notebook 根据每个截面日的历史中证800成分区间构建股票池，并剔除截面日 ST/PT、下一市场交易日停牌及涨跌停股票。`T+1` 来自全市场交易日历，不使用个股自身的下一条记录。

输出数据同时保留 EP、BP、SP、行业和流通市值自然对数，供后续去极值、标准化、中性化及 IC 计算使用。

## 1. 参数与必要字段

源数据只读取股票池筛选和估值因子分析所需的列。回测起点不能早于日行情 Parquet 的实际起点。

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(r"D:\因子分析\数据")#可以直接进行文件操作的路径对象
DAILY_PATH = DATA_DIR / "RESSET_DRESSTK_2016_2025.parquet"
CSI800_PATH = DATA_DIR / "中证800" / "RESSET_IDXCOMPO_1.xlsx"
OUTPUT_PATH = DATA_DIR / "csi800_tradable_pool_2016_2025.parquet"

START = pd.Timestamp("2016-01-04")#转化成时间变量
END = pd.Timestamp("2025-12-30")#给出起始日期

CODE = "股票代码_Stkcd"
DATE = "日期_Date"
STATE = "上市状态_Listedstate"
CLOSE = "收盘价(元)_Clpr"
VOLUME = "成交量(股)_Trdvol"
AMOUNT = "成交金额(元)_Trdsum"
RETURN = "日收益率_Dret"
FLOAT_SHARES = "流通股(股)_Trdshr"
INDUSTRY = "证监会行业大类代码_Csrciccd2"

NEEDED_COLUMNS = [
    CODE,
    DATE,
    STATE,
    CLOSE,
    VOLUME,
    AMOUNT,
    RETURN,
    "市盈率_PE",
    "市净率_PB",
    "市销率_PS",
    FLOAT_SHARES,
    INDUSTRY,
]
#检查这个文件存不存在
for required_path in [DAILY_PATH, CSI800_PATH]:
    if not required_path.exists():#exists() 是 Path 对象的方法，查看这个文件夹存不存在
        raise FileNotFoundError(f"找不到输入文件：{required_path}")#Python 内置的一种错误类型
#f作用是把变量值插进字符串里
print(f"日行情：{DAILY_PATH}")
print(f"成分股：{CSI800_PATH}")

日行情：D:\因子分析\数据\RESSET_DRESSTK_2016_2025.parquet
成分股：D:\因子分析\数据\中证800\RESSET_IDXCOMPO_1.xlsx


## 2. 辅助函数

In [2]:
def code6(series):
    # 股票代码统一为六位字符串。
    result = (#series是序列
        pd.to_numeric(series, errors="coerce")#缺失值改成NAN
        .astype("Int64")#整数型可容纳缺失值，将字符串的改为NAN
        .astype("string")#把整数转换成字符串
        .str.zfill(6)#字符串长度不及六位前面补0，str是series的字符串接口，方便使用字符串的函数
    )
    return result.astype("string[pyarrow]")#按字符串处理，但底层由 PyArrow 来存储


def is_st_pt(series):
    # 识别 ST、*ST、SST、S*ST、PT 等历史状态。
    status = series.astype("string").str.upper().str.strip()#strip的用法是字符串两端字符的空白删掉
    return status.str.contains(r"ST|PT", regex=True, na=False)#回一个布尔型 Series，regex当成正则表达式是否开启，|是或者的用法，缺失值直接当成false


def expand_membership(component_df, trading_dates):
    # 将成分股有效区间展开为股票-交易日明细，结束日期按包含当日处理。
    pieces = []#列表可以放任何元素

    for stock_code, begin, end in component_df[
        [CODE, "_beg", "_end"]
    ].itertuples(index=False, name=None):#一行一行变成元组，index是false不返回行索引
        left = trading_dates.searchsorted(begin, side="left")#side的方法是决定插到左边还是右边
        #应该插到第几个位置，才能继续保持有序，不会插入进去
        #第一个 >= begin 的交易日位置。
        right = (
            len(trading_dates)
            if pd.isna(end)#end是none的话就返回true
            else trading_dates.searchsorted(end, side="left")
        )

        if left < right:
            pieces.append(
                pd.DataFrame({
                    CODE: stock_code,
                    DATE: trading_dates[left:right],
                })
            )#这一步确定日期

    if not pieces:
        raise ValueError("回测区间内没有匹配到中证800成分股。")

    member_days = pd.concat(pieces, ignore_index=True)#接收一个“装着多个 pandas 对象的容器”，然后把里面这些对象拼起来，忽视原有的索引
    member_days[CODE] = member_days[CODE].astype("string[pyarrow]")#用 Arrow 方式存储

    if member_days.duplicated([CODE, DATE]).any():#duplicad里面放列返回series的bool，any看这个series是不是至少有一个true
        raise ValueError("中证800成分股日期区间存在重叠。")

    return member_days


def round_half_up_to_cent(values):
    # 正数价格按四舍五入保留两位小数。
    return np.floor(values * 100 + 0.5 + 1e-10) / 100#向下取整

## 3. 读取必要列并建立市场交易日历

首次出现日期用于识别数据覆盖期内的新上市股票。它是基于完整日行情的近似上市日期；如以后取得官方上市日期，可直接替换该映射。

In [3]:
daily = pd.read_parquet(DAILY_PATH, columns=NEEDED_COLUMNS)
daily[DATE] = pd.to_datetime(daily[DATE], errors="coerce")
daily[CODE] = code6(daily[CODE])

if daily[DATE].isna().any() or daily[CODE].isna().any():#存在问题
    raise ValueError("日行情中存在无法解析的日期或股票代码。")

if daily.duplicated([CODE, DATE]).any():
    duplicate_sample = daily.loc[
        daily.duplicated([CODE, DATE], keep=False),
        [CODE, DATE],#行条件，列选择，这个keep全为true
    ].head()
    raise ValueError(f"发现重复的股票-日期记录：\n{duplicate_sample}")

# 停牌日状态可能为空，只使用该股票过去已知状态向前填充，不使用未来状态。
daily = daily.sort_values([CODE, DATE]).reset_index(drop=True)#先排序，在生成新的索引
daily[STATE] = daily.groupby(CODE, observed=True)[STATE].ffill()#forward fill，，默认升序
#observed=ture只对数据里“实际出现过的类别”进行分组

# 综合数据包含部分非交易日期；只有全市场至少一只股票正常成交才算市场交易日。
has_market_trade = (
    daily[VOLUME].gt(0)#大于0
    & daily[AMOUNT].gt(0)
    & daily[CLOSE].notna()
).groupby(daily[DATE]).any()#groupby() 不一定非得按“自己内部的一列”分组，也可以传入另一个长度和 index 对得上的 Series 作为分组标签
#分组依据 DATE 会变成结果的 index 
all_trading_dates = pd.DatetimeIndex(
    has_market_trade[has_market_trade].index
).sort_values()#pd.datetimeindex转成 pandas 专门的日期索引类型
#自己套自己，因为自己是bool
dataset_start = all_trading_dates.min()

# 以股票首次出现正成交的日期近似其上市首个交易日。
is_stock_trade = (
    daily[VOLUME].gt(0)
    & daily[AMOUNT].gt(0)
    & daily[CLOSE].notna()
)
first_seen_date = (
    daily.loc[is_stock_trade]
    .groupby(CODE, observed=True)[DATE]
    .min()
)#用code索引第一个记录日

calendar = pd.DataFrame({DATE: all_trading_dates})#新建一个 DataFrame，列名叫 DATE，这一列的数据就是 all_trading_dates
calendar["T_plus_1"] = calendar[DATE].shift(-1)#全市场下一交易日
calendar = calendar.loc[
    calendar[DATE].between(START, END)
    & calendar["T_plus_1"].notna()#去掉没有下一交易日的最后一行
].copy()

if calendar.empty:#属性
    raise ValueError("指定回测区间不在日行情数据覆盖范围内。")

required_dates = pd.Index(
    pd.concat([calendar[DATE], calendar["T_plus_1"]]).unique()#默认就是上下拼接，也就是按“行”拼接
)
daily = daily.loc[daily[DATE].isin(required_dates)].copy()#重新指定

print(f"行情记录数：{len(daily):,}")
print(f"市场交易日范围：{all_trading_dates.min().date()} 至 {all_trading_dates.max().date()}")#date是取日期部分，去掉具体时间
print(f"截面日期数：{len(calendar):,}")

行情记录数：10,191,397
市场交易日范围：2016-01-04 至 2025-12-31
截面日期数：2,429


## 4. 筛选历史中证800成分股与 T 日 ST/PT

In [4]:
components = pd.read_excel(
    CSI800_PATH,
    usecols=[
        "指数代码_IdxCd",
        "证券代码_成分_SecuCd_Compo",
        "开始日期_BegDt",
        "结束日期_EndDt",
    ],#决定用什么列
    dtype={"证券代码_成分_SecuCd_Compo": "string"},#某列进来的数据类型
)

components = components.loc[
    pd.to_numeric(components["指数代码_IdxCd"], errors="coerce").eq(906)#更加稳健
].copy()#loc里面返回series的bool，然后把true的全拿出来，copy是为了独立于原对象
components[CODE] = code6(components["证券代码_成分_SecuCd_Compo"])#补0
components["_beg"] = pd.to_datetime(
    components["开始日期_BegDt"], errors="coerce"
)#todatetime也能用在series身上
components["_end"] = pd.to_datetime(
    components["结束日期_EndDt"], errors="coerce"
)

if components["_beg"].isna().any():#存在一个true就报错
    raise ValueError("成分股文件存在无法解析的开始日期。")

member_days = expand_membership(
    components,
    pd.DatetimeIndex(calendar[DATE]),
)#当天有行情 + 当天属于中证800
#先找交易日
base = daily.merge(
    calendar,
    on=DATE,
    how="inner",
    validate="many_to_one",#左边允许重复，右边唯一，匹配补不上就删除
)
#确定中证800
base = base.merge(
    member_days,
    on=[CODE, DATE],
    how="inner",
    validate="many_to_one",
)

if base[STATE].isna().any():
    raise ValueError("部分中证800股票在截面日缺少历史上市状态。")

base["is_st_pt_t"] = is_st_pt(base[STATE])#bool

print(f"历史中证800记录数：{len(base):,}")
print(f"其中 T 日 ST/PT 记录数：{int(base['is_st_pt_t'].sum()):,}")

C:\Users\huang\AppData\Local\Programs\Python\Python311\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


历史中证800记录数：1,885,600
其中 T 日 ST/PT 记录数：12,072


## 5. 对齐真正的 T+1 并识别不可交易状态

In [5]:
t1_quotes = daily[
    [CODE, DATE, STATE, CLOSE, VOLUME, AMOUNT, RETURN]
].rename(columns={
    DATE: "T_plus_1",
    STATE: "state_t1",
    CLOSE: "close_t1",
    VOLUME: "volume_t1",
    AMOUNT: "amount_t1",
    RETURN: "return_t1",
})#更改列的名字

base = base.merge(
    t1_quotes,
    on=[CODE, "T_plus_1"],
    how="left",
    validate="one_to_one",#根据业务逻辑两边都是唯一
)#存好下一日的内容，左边主表

base["is_suspended_t1"] = (
    base["close_t1"].isna()
    | base["volume_t1"].isna()
    | base["amount_t1"].isna()
    | base["volume_t1"].le(0)
    | base["amount_t1"].le(0)
)#检查t1是否合适，生成新的列，返回的是series ，看是不是停牌了

base["board"] = np.select(#多条件版的 if / elif / else
    [
        base[CODE].str.startswith(("300", "301")),
        base[CODE].str.startswith(("688", "689")),
    ],
    ["CHINEXT", "STAR"],
    default="MAIN",
)

unsupported_board = base[CODE].str.startswith(("4", "8", "9"))
if unsupported_board.any():
    bad_codes = base.loc[unsupported_board, CODE].drop_duplicates().head(10).tolist()#转换成列表
    raise ValueError(f"存在未配置涨跌停规则的证券代码：{bad_codes}")

base["listing_date_proxy"] = base[CODE].map(first_seen_date)#根据每一行的股票代码去 first_seen_date 里面查对应日期
base["is_new_stock_in_dataset"] = base["listing_date_proxy"].gt(dataset_start)#在数据集新出现的，bool

trade_day_number = pd.Series(
    np.arange(len(all_trading_dates)),#日期作为 index，交易日序号作为 value
    index=all_trading_dates,
)#方便计算相差交易日
base["_listing_day_no"] = base["listing_date_proxy"].map(trade_day_number)#也是序号
base["_t1_day_no"] = base["T_plus_1"].map(trade_day_number)#算当下t+1的序号

base["no_limit_days"] = np.select(
    [
        base["board"].eq("STAR"),
        base["board"].eq("CHINEXT")
        & base["listing_date_proxy"].ge(pd.Timestamp("2020-08-24")),
        base["board"].eq("MAIN")
        & base["listing_date_proxy"].ge(pd.Timestamp("2023-04-10")),
    ],
    [5, 5, 5],
    default=1,
)#不同的时间点新股上市涨跌幅不同

base["is_no_limit_period_t1"] = (
    base["is_new_stock_in_dataset"]
    & (
        base["_t1_day_no"] - base["_listing_day_no"]
        < base["no_limit_days"]
    )
)#判断t+1这天是否处在无涨跌限制

# 若股票在 T+1 变为 ST/PT，则 T+1 的涨跌幅限制按 5% 处理。
base["is_st_pt_t1"] = is_st_pt(base["state_t1"])#bool
base["limit_rate"] = np.select(
    [
        base["is_st_pt_t1"],
        base["board"].eq("STAR"),
        base["board"].eq("CHINEXT")
        & base["T_plus_1"].ge(pd.Timestamp("2020-08-24")),
    ],
    [0.05, 0.20, 0.20],
    default=0.10,
)

# 依照题目定义，使用 T 日未复权收盘价计算 T+1 理论限价。
base["up_limit_t1"] = round_half_up_to_cent(
    base[CLOSE] * (1 + base["limit_rate"])
)#上
base["down_limit_t1"] = round_half_up_to_cent(
    base[CLOSE] * (1 - base["limit_rate"])
)#下

base["is_up_limit_t1"] = (
    ~base["is_no_limit_period_t1"]#~的意思是取反，true就为false
    & np.isclose(
        base["close_t1"],
        base["up_limit_t1"],
        atol=0.00501,
        rtol=0,
    )#相差不过0.005
)
base["is_down_limit_t1"] = (
    ~base["is_no_limit_period_t1"]
    & np.isclose(
        base["close_t1"],
        base["down_limit_t1"],
        atol=0.00501,
        rtol=0,
    )#跌停同理
)
base["is_limit_t1"] = base["is_up_limit_t1"] | base["is_down_limit_t1"]#任一个都排除

print(f"T+1 停牌或无行情：{int(base['is_suspended_t1'].sum()):,}")
print(f"T+1 涨跌停：{int(base['is_limit_t1'].sum()):,}")

T+1 停牌或无行情：31,136
T+1 涨跌停：18,570


## 6. 生成估值因子基础数据并保存

In [6]:
pool = base.loc[
    ~base["is_st_pt_t"]
    & ~base["is_suspended_t1"]
    #& ~base["is_limit_t1"]，先防止未来信息泄露
].copy()#生成池子

pool["流通市值"] = pool[CLOSE] * pool[FLOAT_SHARES]#对应相乘
pool["ln_流通市值"] = np.log(
    pool["流通市值"].where(pool["流通市值"].gt(0))#series里面用where来判断
)

pool["EP"] = 1 / pool["市盈率_PE"].replace(0, np.nan)#把 市盈率_PE 这一列里所有等于 0 的值，替换成 NaN
pool["BP"] = 1 / pool["市净率_PB"].replace(0, np.nan)
pool["SP"] = 1 / pool["市销率_PS"].replace(0, np.nan)
#市值 / 净利润，市值 / 净资产
pool[["EP", "BP", "SP"]] = pool[["EP", "BP", "SP"]].replace(
    [np.inf, -np.inf], np.nan
)#无穷为nan

pool = pool[
    [
        DATE,
        "T_plus_1",
        CODE,
        STATE,
        "EP",
        "BP",
        "SP",
        "流通市值",
        "ln_流通市值",
        INDUSTRY,
        "return_t1",
    ]
].rename(columns={
    STATE: "T日上市状态",
    INDUSTRY: "行业代码",
    "return_t1": "T_plus_1收益率",
})

pool = pool.sort_values([DATE, CODE]).reset_index(drop=True)#顺手排序
pool.to_parquet(OUTPUT_PATH, index=False, compression="zstd")#生成记录文件

print(f"最终记录数：{len(pool):,}")
print(f"输出文件：{OUTPUT_PATH}")

最终记录数：1,842,543
输出文件：D:\因子分析\数据\csi800_tradable_pool_2016_2025.parquet


## 7. 结果检查

In [7]:
daily_pool_count = pool.groupby(DATE)[CODE].nunique()#先按日期分组在看股票这列在unique

checks = pd.Series({
    "截面日期数": pool[DATE].nunique(),
    "股票代码数": pool[CODE].nunique(),
    "重复股票-日期数": pool.duplicated([CODE, DATE]).sum(),#bool的series直接相加
    "ST/PT 残留数": int(is_st_pt(pool["T日上市状态"]).sum()),
    "T+1 收益率缺失数": pool["T_plus_1收益率"].isna().sum(),
    "非正流通市值数": pool["流通市值"].le(0).sum(),#小于等于
}, name="检查结果")#左index，右value

display(checks.to_frame())#保留index不变
display(daily_pool_count.describe().to_frame("每日股票数"))#describe是描述性统计
display(pool.head())

,检查结果
截面日期数,2429
股票代码数,1371
重复股票-日期数,0
ST/PT 残留数,0
T+1 收益率缺失数,0
非正流通市值数,0


,每日股票数
count,2429.000000
mean,758.560313
std,24.770997
min,708.000000
25%,741.000000
50%,756.000000
75%,782.000000
max,796.000000


,日期_Date,T_plus_1,股票代码_Stkcd,T日上市状态,EP,BP,SP,流通市值,ln_流通市值,行业代码,T_plus_1收益率
0,2016-01-04,2016-01-05,000001,Norm,0.134953,0.877193,0.427350,1.621173e+11,25.811586,J66,0.0062
1,2016-01-04,2016-01-05,000006,Norm,0.029744,0.278552,0.149925,1.401295e+10,23.363248,K70,-0.0328
2,2016-01-04,2016-01-05,000009,Norm,0.030665,0.198807,0.149477,2.572846e+10,23.970863,S90,-0.0470
3,2016-01-04,2016-01-05,000012,Norm,0.021354,0.330033,0.303030,1.577927e+10,23.481963,C30,-0.0283
4,2016-01-04,2016-01-05,000021,Norm,0.011317,0.294985,0.925926,1.596316e+10,23.493550,C39,-0.0627


# 单因子预处理、中性化与 IC 检验

本节直接读取前面生成的可交易中证 800 股票池，并且只读取因子检验所需的列。
每个交易日分别处理 `EP`、`BP`、`SP`，最终使用中性化残差与真正的
T+1 收益率计算 Pearson IC。

## 8. 截面去极值、标准化和缺失值处理

设某日某因子的截面中位数为 $m$，中位数绝对偏差为

$$MAD=\operatorname{median}(|x_i-m|).$$

图片中省略的 $+5/-5$ 按常用的 **5 倍 MAD 缩尾**实现：

$$x_i^{w}=m+\operatorname{clip}\left(\frac{x_i-m}{MAD},-5,5\right)MAD.$$

这等价于把原始因子限制在 $[m-5MAD,\ m+5MAD]$。随后按当日截面均值和
总体标准差（`ddof=0`）进行 Z-score 标准化；原始缺失值在标准化后统一填 0，
表示中性暴露。当截面 `MAD=0` 时不做缩尾，当标准差为 0 时该截面因子全部置 0。

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

FACTOR_DATA_DIR = Path(r"D:\因子分析\数据")
FACTOR_POOL_PATH = FACTOR_DATA_DIR / "csi800_tradable_pool_2016_2025.parquet"
NEUTRAL_FACTOR_PATH = FACTOR_DATA_DIR / "factor_neutralized_2016_2025.parquet"
IC_PATH = FACTOR_DATA_DIR / "factor_ic_2016_2025.parquet"
IC_SUMMARY_PATH = FACTOR_DATA_DIR / "factor_ic_summary_2016_2025.csv"
REGRESSION_DIAGNOSTICS_PATH = (
    FACTOR_DATA_DIR / "factor_neutralization_diagnostics_2016_2025.parquet"
)

FACTOR_DATE = "日期_Date"
FACTOR_CODE = "股票代码_Stkcd"
FACTOR_INDUSTRY = "行业代码"
FACTOR_LOG_MV = "ln_流通市值"
FORWARD_RETURN = "T_plus_1收益率"
FACTOR_COLUMNS = ["EP", "BP", "SP"]
MAD_MULTIPLIER = 5.0
MIN_IC_OBS = 30

FACTOR_READ_COLUMNS = [
    FACTOR_DATE,
    FACTOR_CODE,
    *FACTOR_COLUMNS,#相当于解包
    FACTOR_LOG_MV,
    FACTOR_INDUSTRY,
    FORWARD_RETURN,
]

if not FACTOR_POOL_PATH.exists():
    raise FileNotFoundError(f"找不到股票池文件：{FACTOR_POOL_PATH}")

# 只读取预处理、中性化和 IC 计算实际需要的 8 列。
factor_data = pd.read_parquet(
    FACTOR_POOL_PATH,
    columns=FACTOR_READ_COLUMNS,
)
factor_data[FACTOR_DATE] = pd.to_datetime(
    factor_data[FACTOR_DATE], errors="coerce"
)
numeric_columns = [*FACTOR_COLUMNS, FACTOR_LOG_MV, FORWARD_RETURN]
factor_data[numeric_columns] = factor_data[numeric_columns].replace(
    [np.inf, -np.inf], np.nan
)
factor_data = factor_data.sort_values(
    [FACTOR_DATE, FACTOR_CODE]
).reset_index(drop=True)

if factor_data[FACTOR_DATE].isna().any():
    raise ValueError("股票池中存在无法解析的日期。")
if factor_data.duplicated([FACTOR_DATE, FACTOR_CODE]).any():
    raise ValueError("股票池中存在重复的股票-日期记录。")
if factor_data[FORWARD_RETURN].isna().any():
    raise ValueError("股票池中存在缺失的 T+1 收益率。")

def mad_clip(series, multiplier=MAD_MULTIPLIER):
    """按截面中位数 +/- multiplier * MAD 对有效值缩尾。"""
    values = pd.to_numeric(series, errors="coerce").astype(float)
    median = values.median(skipna=True)
    mad = (values - median).abs().median(skipna=True)

    if not np.isfinite(median) or not np.isfinite(mad) or mad <= 0:
        return values#有问题直接原始正常数值

    return values.clip(
        lower=median - multiplier * mad,
        upper=median + multiplier * mad,
    )#用来压缩上下限，超过或低于强行设置

def population_zscore(series):
    """按截面总体标准差标准化；缺失值留到下一步统一填 0。"""
    values = pd.to_numeric(series, errors="coerce").astype(float)
    mean = values.mean(skipna=True)
    std = values.std(skipna=True, ddof=0)#自由度是n

    if not np.isfinite(mean) or not np.isfinite(std) or std <= 0:
        return pd.Series(np.nan, index=series.index, dtype=float)

    return (values - mean) / std

print(f"读取记录数：{len(factor_data):,}")
print(f"实际读取列：{FACTOR_READ_COLUMNS}")
print(
    f"截面范围：{factor_data[FACTOR_DATE].min().date()} 至 "
    f"{factor_data[FACTOR_DATE].max().date()}"
)

读取记录数：1,842,543
实际读取列：['日期_Date', '股票代码_Stkcd', 'EP', 'BP', 'SP', 'ln_流通市值', '行业代码', 'T_plus_1收益率']
截面范围：2016-01-04 至 2025-12-30


In [9]:
preprocess_rows = []
winsor_columns = []
zscore_columns = []

for factor in FACTOR_COLUMNS:
    winsor_column = f"{factor}_MAD缩尾"
    zscore_column = f"{factor}_标准化"
    winsor_columns.append(winsor_column)
    zscore_columns.append(zscore_column)

    factor_data[winsor_column] = (
        factor_data.groupby(FACTOR_DATE, sort=False, observed=True)[factor]
        .transform(mad_clip)#transform的作用是对每一个series都调用mad_clip这个函数
    )
    factor_data[zscore_column] = (
        factor_data.groupby(FACTOR_DATE, sort=False, observed=True)[winsor_column]
        .transform(population_zscore)
        .fillna(0.0)#缺失值填充0
    )

    original = pd.to_numeric(factor_data[factor], errors="coerce").to_numpy(float)#变成numpy数组
    winsorized = factor_data[winsor_column].to_numpy(float)
    finite_pair = np.isfinite(original) & np.isfinite(winsorized)#原始值和缩尾后值都有效的那些位置
    clipped_count = int(np.sum(
        finite_pair
        & ~np.isclose(original, winsorized, rtol=0, atol=1e-15)#看原始值和所谓后值一样，取反就不同了
    ))#看更改的数量，相对误差，绝对误差

    preprocess_rows.append({
        "因子": factor,
        "原始缺失数": int(factor_data[factor].isna().sum()),
        "MAD缩尾数": clipped_count,
        "标准化后缺失数": int(factor_data[zscore_column].isna().sum()),
        "标准化后零值数": int(factor_data[zscore_column].eq(0).sum()),
    })#往 preprocess_rows 这个列表里追加一个字典

preprocess_summary = pd.DataFrame(preprocess_rows).set_index("因子")#因子变成索引
factor_data = factor_data.drop(columns=winsor_columns)#把缩尾列删掉

display(preprocess_summary)
display(
    factor_data.groupby(FACTOR_DATE, observed=True)[zscore_columns]
    .mean()#先取均值
    .abs()#再取绝对值
    .max()#再取最大值
    .rename("每日标准化因子均值绝对值最大值")#命名
    .to_frame()#在变成dataframe
)

,原始缺失数,MAD缩尾数,标准化后缺失数,标准化后零值数
因子,,,,
EP,934,137835,0,934
BP,11131,95942,0,11131
SP,20150,210874,0,20150


,每日标准化因子均值绝对值最大值
EP_标准化,3.152267e-16
BP_标准化,4.228953e-16
SP_标准化,3.128810e-16


## 9. 流通市值与行业中性化

每个交易日分别估计：

$$z_{i,T}=\alpha_T+\beta_T\ln(\text{流通市值}_{i,T})
+\sum_k\gamma_{k,T}I_{i,k,T}+\varepsilon_{i,T}.$$

回归含截距，行业虚拟变量删除一个基准类别，行业缺失值作为独立的 `UNKNOWN`
类别。为改善数值稳定性，回归前仅对 `ln(流通市值)` 在当日截面内标准化，
这不会改变残差。市值缺失记录不参与回归和 IC，不对控制变量做人为填补。
最终因子值为回归残差 $\varepsilon_{i,T}$。

In [10]:
neutral_columns = [f"{factor}_中性化" for factor in FACTOR_COLUMNS]#先生成中性化的列

def neutralize_cross_sections(data, y_columns):#中性化
    residuals = np.full((len(data), len(y_columns)), np.nan, dtype=float)#形状，再加上用什么填充
    diagnostics = []
    grouped_positions = data.groupby(
        FACTOR_DATE, sort=True, observed=True
    ).indices#indices把 groupby 之后每个组对应的“原始行位置”取出来，返回一个类似字典。key是分组，value是位置

    for trade_date, positions in grouped_positions.items():#字典，同时拿key和values
        positions = np.asarray(positions, dtype=np.int64)#方便bool索引
        cross_section = data.iloc[positions]#适配iloc用法
        log_mv_all = pd.to_numeric(
            cross_section[FACTOR_LOG_MV], errors="coerce"
        ).to_numpy(float)
        valid_control = np.isfinite(log_mv_all)#bool数组
        valid_positions = positions[valid_control]#返回正常的
        n_obs = int(valid_control.sum())

        diagnostic = {
            FACTOR_DATE: pd.Timestamp(trade_date),
            "截面总数": len(positions),
            "回归样本数": n_obs,
            "市值缺失数": int((~valid_control).sum()),
        }

        if n_obs < MIN_IC_OBS:
            diagnostic.update({
                "设计矩阵列数": np.nan,
                "矩阵秩": np.nan,
                "状态": "样本不足",
            })#用另一个字典的内容，批量添加或修改当前字典
            diagnostics.append(diagnostic)#append直接加
            continue
        #没问题下一步
        valid_cross_section = cross_section.loc[valid_control]#loc支持numpy数组和list
        log_mv = log_mv_all[valid_control]
        log_mv_std = log_mv.std(ddof=0)#总体标准差
        if not np.isfinite(log_mv_std) or log_mv_std <= 0:
            diagnostic.update({
                "设计矩阵列数": np.nan,
                "矩阵秩": np.nan,
                "状态": "市值无波动",
            })
            diagnostics.append(diagnostic)
            continue

        log_mv_z = (log_mv - log_mv.mean()) / log_mv_std
        industry = (
            valid_cross_section[FACTOR_INDUSTRY]
            .astype("string")
            .fillna("UNKNOWN")
        )#行业便字符串再填充缺失为nan
        industry_dummies = pd.get_dummies(
            industry,
            drop_first=True,#避免多重共线性
            dtype=float,
        )#热编码
        design = np.column_stack([
            np.ones(n_obs, dtype=float),#常数项
            log_mv_z,#市值
            industry_dummies.to_numpy(float),#行业热编码
        ])#设计矩阵，返回的是二维numpy array
        targets = valid_cross_section[y_columns].to_numpy(float)#y

        if n_obs <= design.shape[1]:
            diagnostic.update({
                "设计矩阵列数": design.shape[1],#0行1列
                "矩阵秩": np.nan,
                "状态": "自由度不足",
            })
            diagnostics.append(diagnostic)
            continue

        coefficients, _, rank, _ = np.linalg.lstsq(#NumPy 点 linear algebra 点 least squares
            design,
            targets,
            rcond=None,
        )#_表示不需要，系数，秩
        residuals[valid_positions, :] = targets - design @ coefficients#@是矩阵乘法
        #前面行索引后面所有列

        diagnostic.update({
            "设计矩阵列数": design.shape[1],
            "矩阵秩": int(rank),
            "状态": "正常" if rank == design.shape[1] else "秩不足",
        })
        diagnostics.append(diagnostic)

    return residuals, pd.DataFrame(diagnostics)#返回修改后的残差和诊断

neutral_values, regression_diagnostics = neutralize_cross_sections(
    factor_data,
    zscore_columns,
)#中性化后得知
factor_data[neutral_columns] = neutral_values

display(
    regression_diagnostics["状态"]
    .value_counts(dropna=False)#计数
    .rename("截面数")
    .to_frame()
)
display(
    regression_diagnostics[
        ["截面总数", "回归样本数", "市值缺失数", "设计矩阵列数"]#每天行业数量可能不同
    ].describe()
)#看截面缺不缺失

,截面数
状态,
正常,2429


,截面总数,回归样本数,市值缺失数,设计矩阵列数
count,2429.000000,2429.000000,2429.000000,2429.000000
mean,758.560313,758.041169,0.519144,69.065047
std,24.770997,24.811164,0.957576,1.254272
min,708.000000,708.000000,0.000000,67.000000
25%,741.000000,741.000000,0.000000,68.000000
50%,756.000000,755.000000,0.000000,69.000000
75%,782.000000,782.000000,1.000000,70.000000
max,796.000000,796.000000,7.000000,72.000000


## 10. 计算每日 IC 与 ICIR

对每个交易日，以中性化残差和对应股票的 T+1 收益率计算 Pearson 相关系数。
有效配对样本少于 30、因子无截面波动或收益率无截面波动时，该日 IC 记为缺失。
`ICIR` 按 `IC均值 / IC标准差` 计算，不做年化。

In [11]:
def pearson_ic(factor_values, forward_returns, min_obs=MIN_IC_OBS):
    factor_array = pd.to_numeric(
        factor_values, errors="coerce"
    ).to_numpy(float)
    return_array = pd.to_numeric(
        forward_returns, errors="coerce"
    ).to_numpy(float)
    valid = np.isfinite(factor_array) & np.isfinite(return_array)#都要有效

    n_obs = int(valid.sum())
    if n_obs < min_obs:
        return np.nan, n_obs

    x = factor_array[valid]
    y = return_array[valid]
    if x.std(ddof=0) <= 0 or y.std(ddof=0) <= 0:
        return np.nan, n_obs

    return float(np.corrcoef(x, y)[0, 1]), n_obs#np.corrcoef返回相关系数矩阵因此需要给出索引

ic_rows = []
grouped_positions = factor_data.groupby(
    FACTOR_DATE, sort=True, observed=True
).indices#返回的是整数位置，分组按顺序返回，只分类存在的

for trade_date, positions in grouped_positions.items():
    cross_section = factor_data.iloc[np.asarray(positions, dtype=np.int64)]
    for factor, neutral_column in zip(FACTOR_COLUMNS, neutral_columns):#多个序列中相同位置的元素一一配对
        ic_value, n_obs = pearson_ic(
            cross_section[neutral_column],
            cross_section[FORWARD_RETURN],#未来收益率
        )
        ic_rows.append({
            FACTOR_DATE: pd.Timestamp(trade_date),
            "因子": factor,
            "IC": ic_value,
            "有效样本数": n_obs,
        })

ic_series = pd.DataFrame(ic_rows).sort_values(
    ["因子", FACTOR_DATE]
).reset_index(drop=True)#重新设置索引从0开始

ic_group = ic_series.groupby("因子", sort=False)["IC"]#每个组后面跟着ic这个列表，允许先产生分组对象：factor，values
ic_summary = pd.DataFrame({
    "有效期数": ic_group.count(),
    "IC均值": ic_group.mean(),
    "IC标准差": ic_group.std(ddof=1),
    "IC绝对值均值": ic_group.apply(lambda values: values.abs().mean()),#lambda 参数: 返回结果
    "正IC比例": ic_group.apply(lambda values: values.gt(0).mean()),
}).reindex(FACTOR_COLUMNS)#ic_group 的每个统计结果，本身就已经以因子名称作为索引
ic_summary["ICIR"] = (
    ic_summary["IC均值"] / ic_summary["IC标准差"]#信息系数比率，也常写作 IC 信息比率
)
ic_summary["IC均值t值"] = (
    ic_summary["IC均值"]
    / (ic_summary["IC标准差"] / np.sqrt(ic_summary["有效期数"]))
)#大样本性质

neutral_output_columns = [
    FACTOR_DATE,
    FACTOR_CODE,
    FORWARD_RETURN,
    *zscore_columns,
    *neutral_columns,
]
factor_data[neutral_output_columns].to_parquet(
    NEUTRAL_FACTOR_PATH,
    index=False,
    compression="zstd",
)#存储出去
ic_series.to_parquet(IC_PATH, index=False, compression="zstd")
regression_diagnostics.to_parquet(
    REGRESSION_DIAGNOSTICS_PATH,
    index=False,#不把 DataFrame 的行索引保存到文件
    compression="zstd",#压缩算法压缩 Parquet 文件
)
ic_summary.to_csv(IC_SUMMARY_PATH, encoding="utf-8-sig")

display(ic_summary)
display(ic_series.head(9))
print(f"中性化因子：{NEUTRAL_FACTOR_PATH}")
print(f"IC 时间序列：{IC_PATH}")
print(f"IC 汇总：{IC_SUMMARY_PATH}")

,有效期数,IC均值,IC标准差,IC绝对值均值,正IC比例,ICIR,IC均值t值
因子,,,,,,,
EP,2429,0.010934,0.073061,0.058364,0.543022,0.149652,7.375564
BP,2429,0.002211,0.085386,0.068353,0.489090,0.025890,1.275964
SP,2429,0.002813,0.079549,0.062804,0.502676,0.035361,1.742771


,日期_Date,因子,IC,有效样本数
0,2016-01-04,BP,0.108326,743
1,2016-01-05,BP,0.088422,746
2,2016-01-06,BP,0.149952,745
3,2016-01-07,BP,0.162880,744
4,2016-01-08,BP,0.177562,743
5,2016-01-11,BP,0.051212,745
6,2016-01-12,BP,0.158437,744
7,2016-01-13,BP,-0.169739,743
8,2016-01-14,BP,-0.137499,745


中性化因子：D:\因子分析\数据\factor_neutralized_2016_2025.parquet
IC 时间序列：D:\因子分析\数据\factor_ic_2016_2025.parquet
IC 汇总：D:\因子分析\数据\factor_ic_summary_2016_2025.csv


## 11. 结果校验

下列检查验证数据粒度、回归成功率、残差与控制变量的正交性以及 IC 边界。
市值缺失记录应当与中性化残差缺失记录一致，这是主动排除缺失控制变量的结果。

In [12]:
valid_for_regression = factor_data[FACTOR_LOG_MV].notna()
missing_mv_rows = int((~valid_for_regression).sum())#看缺失了多少
missing_residual_rows = int(
    factor_data[neutral_columns].isna().any(axis=1).sum()#axis=1表示按行，any表示只要有一个true就返回true
)#把多个 Series 组成一个 DataFrame，再整体判断，返回的是bool表格
failed_regressions = int(
    regression_diagnostics["状态"].ne("正常").sum()#ne表示不等于，返回bool series
)

daily_residual_mean_max = (
    factor_data.groupby(FACTOR_DATE, observed=True)[neutral_columns]
    .mean()
    .abs()
    .max()
)#不同日期j的残差均值绝对值最大值，先按日期分组，取均值，再取绝对值，再取最大值

residual_mv_corr_max = {}
for neutral_column in neutral_columns:
    correlations = []
    for _, group in factor_data.loc[valid_for_regression].groupby(#groupby分组名称, 该分组对应的DataFrame
        FACTOR_DATE, observed=True
    ):#日期，股票有效数据
        correlations.append(
            group[neutral_column].corr(group[FACTOR_LOG_MV])
        )
    residual_mv_corr_max[neutral_column] = np.nanmax(
        np.abs(correlations)
    )#忽略nan找最大绝对值
residual_mv_corr_max = pd.Series(
    residual_mv_corr_max,#key为索引
    name="残差与ln流通市值相关系数绝对值最大值",#toframe后列变成这个名字
)

industry_for_check = (
    factor_data[FACTOR_INDUSTRY].astype("string").fillna("UNKNOWN")
)#就是一个series
industry_residual_mean_max = (
    factor_data.loc[valid_for_regression]
    .assign(_行业检查=industry_for_check.loc[valid_for_regression])#新增一列，不会修改原来的 factor_data
    .groupby([FACTOR_DATE, "_行业检查"], observed=True)[neutral_columns]
    .mean()
    .abs()
    .max()
)#先按日期和行业分组，取均值，再取绝对值，再取最大值

ic_out_of_bounds = int((
    ic_series["IC"].notna()
    & ~ic_series["IC"].between(-1, 1, inclusive="both")
).sum())#看看超界的ic有多少

validation_checks = pd.Series({
    "重复股票-日期数": int(
        factor_data.duplicated([FACTOR_DATE, FACTOR_CODE]).sum()
    ),
    "回归失败截面数": failed_regressions,
    "流通市值对数缺失记录数": missing_mv_rows,
    "中性化残差缺失记录数": missing_residual_rows,
    "IC缺失期数": int(ic_series["IC"].isna().sum()),
    "IC越界数": ic_out_of_bounds,
    "每日残差均值绝对值最大值": float(daily_residual_mean_max.max()),
    "残差与市值相关绝对值最大值": float(residual_mv_corr_max.max()),
    "行业内残差均值绝对值最大值": float(
        industry_residual_mean_max.max()
    ),
}, name="检查结果")

if validation_checks["重复股票-日期数"] != 0:
    raise AssertionError("发现重复的股票-日期记录。")
if failed_regressions != 0:
    raise AssertionError("存在未成功完成的截面回归。")
if missing_residual_rows != missing_mv_rows:
    raise AssertionError("残差缺失与市值控制变量缺失不一致。")
if ic_out_of_bounds != 0:
    raise AssertionError("发现超出 [-1, 1] 的 IC。")
if daily_residual_mean_max.max() > 1e-8:
    raise AssertionError("截面残差均值未通过数值精度检查。")
if residual_mv_corr_max.max() > 1e-8:
    raise AssertionError("残差仍与流通市值显著相关。")
if industry_residual_mean_max.max() > 1e-8:
    raise AssertionError("残差仍包含可检测的行业均值差异。")

display(validation_checks.to_frame())
display(daily_residual_mean_max.rename("每日残差均值绝对值最大值").to_frame())
display(residual_mv_corr_max.to_frame())
display(industry_residual_mean_max.rename("行业内残差均值绝对值最大值").to_frame())
print("全部校验通过。")

,检查结果
重复股票-日期数,0.000000e+00
回归失败截面数,0.000000e+00
流通市值对数缺失记录数,1.261000e+03
中性化残差缺失记录数,1.261000e+03
IC缺失期数,0.000000e+00
IC越界数,0.000000e+00
每日残差均值绝对值最大值,2.046365e-15
残差与市值相关绝对值最大值,1.759326e-14
行业内残差均值绝对值最大值,2.944311e-13


,每日残差均值绝对值最大值
EP_中性化,1.846891e-15
BP_中性化,1.579532e-15
SP_中性化,2.046365e-15


,残差与ln流通市值相关系数绝对值最大值
EP_中性化,1.718529e-14
BP_中性化,1.759326e-14
SP_中性化,1.136715e-14


,行业内残差均值绝对值最大值
EP_中性化,2.229883e-13
BP_中性化,2.944311e-13
SP_中性化,2.483939e-13


全部校验通过。


# 沪深300行业约束的单因子指数增强 T+1 开盘回测（2016年试跑）

本节使用真实的沪深300月末成分权重，先完成2016年的可复现试跑。

**本次明确口径**

- 信号期：2016年1月至11月的最后交易日；持仓期从2016年2月首个交易日收盘后开始，到2016年12月30日结束。
- 候选池：中证800可交易池；剔除科创板，信号日剔除ST/PT，入场日无行情或停牌的股票已由前文股票池排除。
- 行业权重：月末沪深300成分股真实个股权重，按该日有效的证监会一级行业加总；权重先在每月归一化。
- 行业缺失：行业无法识别的沪深300权重单列为 `UNKNOWN`，增强组合保留等额现金，不重新分配。
- 选股：EP、BP、SP中性化因子均按降序；每行业选 `ceil(有效候选数 × 20%)`，并按六位股票代码升序稳定处理并列。
- 换仓：下月首个交易日收盘换仓；新组合从该收盘到下一交易日收盘开始承担收益，月内权重自然漂移。
- 收益与费用：使用RESSET `日收益率_Dret`；股票日收益为简单收益率，净值跨日复利；买卖均收0.4%，初始建仓收费，期末不清仓。
- 基准：使用指数行情文件中的沪深300（000300）日收益率；首个调仓日从收盘后开始持有，因此该日基准收益设为0。月末成分权重仍只用于计算行业约束。

In [13]:
from pathlib import Path
import math
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook


plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


BT_DATA_DIR = Path(r"D:\因子分析\数据")
BT_OUTPUT_DIR = Path(
    os.environ.get(
        "FACTOR_BACKTEST_OUTPUT_DIR",
        BT_DATA_DIR / "回测结果_交易约束与滑点版",
    )
)
BT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BT_DAILY_PATH = BT_DATA_DIR / "RESSET_DRESSTK_2016_2025.parquet"
BT_POOL_PATH = BT_DATA_DIR / "csi800_tradable_pool_2016_2025.parquet"
BT_FACTOR_PATH = BT_DATA_DIR / "factor_neutralized_2016_2025.parquet"
BT_WEIGHT_DIR = BT_DATA_DIR / "沪深300"
BT_INDEX_PATH = BT_DATA_DIR / "指数行情" / "common_index_quotes_2016_2025.parquet"
BT_RISK_FREE_PATH = BT_DATA_DIR / "RESSET_BDDRFRET_1.xlsx"

BT_INDUSTRY_WEIGHT_PATH = BT_OUTPUT_DIR / "hs300_industry_weights_2016.csv"
BT_HOLDINGS_PATH = BT_OUTPUT_DIR / "single_factor_monthly_holdings_2016.parquet"
BT_ALLOCATION_AUDIT_PATH = BT_OUTPUT_DIR / "single_factor_allocation_audit_2016.csv"
BT_DAILY_RESULT_PATH = BT_OUTPUT_DIR / "single_factor_daily_backtest_2016.parquet"
BT_TURNOVER_PATH = BT_OUTPUT_DIR / "single_factor_execution_detail_2016.parquet"
BT_END_PENDING_PATH = BT_OUTPUT_DIR / "single_factor_end_pending_orders_2016.csv"
BT_TRADEABILITY_PATH = BT_OUTPUT_DIR / "daily_tradeability_2016.parquet"
BT_PERFORMANCE_PATH = BT_OUTPUT_DIR / "single_factor_performance_2016.csv"
BT_NAV_PLOT_PATH = BT_OUTPUT_DIR / "single_factor_nav_2016.png"

BT_START_CLOSE = pd.Timestamp("2016-02-01")
BT_END = pd.Timestamp("2016-12-30")
BT_SIGNAL_START = pd.Timestamp("2016-01-01")
BT_SIGNAL_END = pd.Timestamp("2016-11-30")
BT_TOP_FRACTION = 0.20
BT_ONE_SIDE_FEE_RATE = 0.004
BT_SLIPPAGE_RATE = 0.0005  # 单边 5bp：买入价上浮、卖出价下浮
BT_ANNUAL_TRADING_DAYS = 252
BT_EXCLUDE_STAR = True

BT_DATE = "日期_Date"
BT_NEXT_DATE = "T_plus_1"
BT_CODE = "股票代码_Stkcd"
BT_RETURN = "日收益率_Dret"
BT_STATE = "上市状态_Listedstate"
BT_PREV_CLOSE = "前收盘价(元)_PrevClPr"
BT_OPEN = "开盘价(元)_Oppr"
BT_HIGH = "最高价(元)_Hipr"
BT_LOW = "最低价(元)_Lopr"
BT_CLOSE = "收盘价(元)_Clpr"
BT_VOLUME = "成交量(股)_Trdvol"
BT_AMOUNT = "成交金额(元)_Trdsum"
BT_DAILY_INDUSTRY = "证监会行业大类代码_Csrciccd2"
BT_POOL_INDUSTRY = "行业代码"

BT_WEIGHT_DATE = "截止日期_EndDt"
BT_WEIGHT_CODE = "成分股代码_CompoStkCd"
BT_WEIGHT_NAME = "成分股名称_CompoStkNm"
BT_WEIGHT_VALUE = "权重(%)_Weight"
BT_WEIGHT_INDEX_CODE = "指数代码_IdxCd"

BT_INDEX_CODE = "指数代码_IdxCd"
BT_INDEX_DATE = "交易日期_TrdDt"
BT_INDEX_RETURN = "日收益率"
BT_INDEX_OPEN = "开盘价(元/点)_OpPr"
BT_INDEX_CLOSE = "收盘价(元/点)_ClPr"
BT_HS300_CODE = "000300"

BT_RF_DATE = "日期_Date"
BT_RF_RETURN = "日无风险收益率_DRFRet"

BT_FACTORS = ["EP", "BP", "SP"]
BT_FACTOR_COLUMNS = {factor: f"{factor}_中性化" for factor in BT_FACTORS}

BT_CSRC_LEVEL1_NAMES = {
    "A": "农、林、牧、渔业",
    "B": "采矿业",
    "C": "制造业",
    "D": "电力、热力、燃气及水生产和供应业",
    "E": "建筑业",
    "F": "批发和零售业",
    "G": "交通运输、仓储和邮政业",
    "H": "住宿和餐饮业",
    "I": "信息传输、软件和信息技术服务业",
    "J": "金融业",
    "K": "房地产业",
    "L": "租赁和商务服务业",
    "M": "科学研究和技术服务业",
    "N": "水利、环境和公共设施管理业",
    "O": "居民服务、修理和其他服务业",
    "P": "教育",
    "Q": "卫生和社会工作",
    "R": "文化、体育和娱乐业",
    "S": "综合",
    "UNKNOWN": "行业缺失（权重保留现金）",
}

for bt_required_path in [
    BT_DAILY_PATH,
    BT_POOL_PATH,
    BT_FACTOR_PATH,
    BT_WEIGHT_DIR,
    BT_INDEX_PATH,
    BT_RISK_FREE_PATH,
]:
    if not bt_required_path.exists():
        raise FileNotFoundError(f"找不到回测输入：{bt_required_path}")

print(f"试跑持仓区间：{BT_START_CLOSE.date()} 收盘后至 {BT_END.date()}")
print(f"单边手续费率：{BT_ONE_SIDE_FEE_RATE:.4%}")
print(f"单边滑点率：{BT_SLIPPAGE_RATE:.4%}")
print(f"输出目录：{BT_OUTPUT_DIR}")

试跑持仓区间：2016-02-01 收盘后至 2016-12-30
单边手续费率：0.4000%
单边滑点率：0.0500%
输出目录：D:\因子分析\数据\回测结果_交易约束与滑点版


## 12. 读取并校验月末沪深300成分权重

自动扫描“沪深300”目录下的权重工作簿，只使用2016年1—11月。2015年12月权重不参与本次从2016年2月开始的试跑；2016年12月权重对应2017年1月调仓，也不参与。

In [14]:
def bt_code6(values):
    return (
        pd.to_numeric(values, errors="coerce")
        .astype("Int64")
        .astype("string")
        .str.zfill(6)#zerofill
    )#一样填充


def bt_level1_industry(values):
    codes = values.astype("string").str.strip().str.upper().str[:1]#strip去掉空格，upper大写，[:1]取第一个字符
    return codes.where(codes.isin(list("ABCDEFGHIJKLMNOPQRS")), "UNKNOWN")#list(...) 会把这个字符串按字符拆开，unkwon是替换值


bt_weight_files = sorted(BT_WEIGHT_DIR.glob("**/RESSET_IDXCOMPOWGH_*.xlsx"))
#递归搜索 BT_WEIGHT_DIR 目录及其所有子目录中，所有名字符合 RESSET_IDXCOMPOWGH_*.xlsx 的 Excel 文件，然后按路径排序
if not bt_weight_files:
    raise FileNotFoundError(f"未找到沪深300月度成分权重文件：{BT_WEIGHT_DIR}")

bt_weight_parts = []
for bt_weight_file in bt_weight_files:
    with warnings.catch_warnings():
        #在这个 with 代码块里面，临时修改 warning 的处理方式；出了这个代码块以后，恢复原来的 warning 设置
        warnings.filterwarnings(
            "ignore",#忽略符合条件的警告。
            message="Workbook contains no default style",#警告信息
            category=UserWarning,#类型
        )#这个警告通常不影响数据读取，所以不想让控制台一直刷出来
        bt_part = pd.read_excel(
            bt_weight_file,
            usecols=[
                BT_WEIGHT_INDEX_CODE,
                BT_WEIGHT_DATE,
                BT_WEIGHT_CODE,
                BT_WEIGHT_NAME,
                BT_WEIGHT_VALUE,
            ],
        )
    bt_part["来源文件"] = str(bt_weight_file)
    bt_weight_parts.append(bt_part)#保存

bt_constituent_weights = pd.concat(bt_weight_parts, ignore_index=True)#拼接起来
bt_constituent_weights[BT_WEIGHT_DATE] = pd.to_datetime(
    bt_constituent_weights[BT_WEIGHT_DATE], errors="coerce"
)#缺失值，not a time，返回NaT
bt_constituent_weights[BT_WEIGHT_CODE] = bt_code6(
    bt_constituent_weights[BT_WEIGHT_CODE]
)#函数填充
bt_constituent_weights[BT_WEIGHT_VALUE] = pd.to_numeric(
    bt_constituent_weights[BT_WEIGHT_VALUE], errors="coerce"
)#变为数字
bt_constituent_weights = bt_constituent_weights.loc[
    pd.to_numeric(bt_constituent_weights[BT_WEIGHT_INDEX_CODE], errors="coerce").eq(300)#数字类型判断
    & bt_constituent_weights[BT_WEIGHT_DATE].between(BT_SIGNAL_START, BT_SIGNAL_END)#时间确定
].copy()

if bt_constituent_weights[[BT_WEIGHT_DATE, BT_WEIGHT_CODE, BT_WEIGHT_VALUE]].isna().any().any():
    #第一个any()axis=0是按列判断，第二个any()是按行判断
    raise ValueError("沪深300权重存在缺失或无法解析的日期、代码、权重。")
if bt_constituent_weights.duplicated([BT_WEIGHT_DATE, BT_WEIGHT_CODE]).any():#重复日期重复代码
    raise ValueError("沪深300权重存在重复的日期-股票记录。")

bt_weight_quality = bt_constituent_weights.groupby(BT_WEIGHT_DATE).agg(#对每个分组，一次性做多个汇总统计
    成分股数=(BT_WEIGHT_CODE, "nunique"),#每个日期里，BT_WEIGHT_CODE 有多少个不同的股票代码
    原始权重合计=(BT_WEIGHT_VALUE, "sum"),#每个日期所有成分股权重加起来
    #新列名=(要统计的列, 用什么函数统计)
)
bt_expected_signal_months = pd.period_range("2016-01", "2016-11", freq="M")
#从 start 到 end，按照 freq 指定的周期，生成一串 Period
bt_actual_signal_months = pd.PeriodIndex(bt_weight_quality.index, freq="M")
#把 bt_weight_quality.index 里面已有的日期，转换成“按月表示的 PeriodIndex”
if not bt_actual_signal_months.equals(bt_expected_signal_months):
    raise AssertionError(
        "权重月份不完整，缺少："
        f"{bt_expected_signal_months.difference(bt_actual_signal_months).tolist()}"#在 expected 里面有，但是在 actual 里面没有的月份
    )
if not bt_weight_quality["成分股数"].eq(300).all():#判断这一整列布尔值是不是全部都是 True
    raise AssertionError("部分信号月的沪深300成分股数不等于300。")
if not bt_weight_quality["原始权重合计"].between(99.9, 100.1).all():
    raise AssertionError("部分信号月的原始权重合计明显偏离100%。")

bt_constituent_weights["是否科创板"] = bt_constituent_weights[
    BT_WEIGHT_CODE
].str.startswith(("688", "689"), na=False)
if BT_EXCLUDE_STAR:#排除科创板
    bt_constituent_weights = bt_constituent_weights.loc[
        ~bt_constituent_weights["是否科创板"]
    ].copy()

bt_monthly_weight_base = bt_constituent_weights.groupby(BT_WEIGHT_DATE)[
    BT_WEIGHT_VALUE
].transform("sum")#返回结果长度和原 DataFrame 一样，每组算出一个统计值，再返回到组里的每一行
bt_constituent_weights["个股权重"] = (
    bt_constituent_weights[BT_WEIGHT_VALUE] / bt_monthly_weight_base
)#重新分配

display(bt_weight_quality)
print(f"读取信号月数：{len(bt_weight_quality)}；权重文件数：{len(bt_weight_files)}")

,成分股数,原始权重合计
截止日期_EndDt,,
2016-01-29,300,100.002
2016-02-29,300,99.992
2016-03-31,300,100.004
2016-04-29,300,100.005
2016-05-31,300,99.999
2016-06-30,300,100.000
2016-07-29,300,99.999
2016-08-31,300,100.000
2016-09-30,300,99.999


读取信号月数：11；权重文件数：13


## 13. 计算每月沪深300行业权重并构建持仓

行业使用信号日行情记录中的证监会行业大类代码首字母。缺失行业不猜测、不丢权重，作为现金额度保留。

In [15]:
bt_pool = pd.read_parquet(
    BT_POOL_PATH,
    columns=[BT_DATE, BT_NEXT_DATE, BT_CODE, BT_POOL_INDUSTRY],
)
bt_pool[BT_DATE] = pd.to_datetime(bt_pool[BT_DATE], errors="coerce")
bt_pool[BT_NEXT_DATE] = pd.to_datetime(bt_pool[BT_NEXT_DATE], errors="coerce")
bt_pool[BT_CODE] = bt_code6(bt_pool[BT_CODE])

bt_signal_dates = pd.DatetimeIndex(bt_weight_quality.index).sort_values()
bt_signal_calendar = (
    bt_pool.loc[bt_pool[BT_DATE].isin(bt_signal_dates), [BT_DATE, BT_NEXT_DATE]]
    .drop_duplicates()#去重
    .rename(columns={BT_DATE: "信号日", BT_NEXT_DATE: "调仓日"})#rename里面用columns={列名：“新列名”}
    .sort_values("信号日")#排序，pandas是sort_values
    .reset_index(drop=True)#新的索引0123
)
if bt_signal_calendar["信号日"].nunique() != len(bt_signal_dates):
    raise AssertionError("候选池未覆盖全部月末权重日期。")
if bt_signal_calendar.duplicated("信号日").any():
    raise AssertionError("同一信号日在候选池中对应多个下一交易日。")
if not (
    bt_signal_calendar["调仓日"].dt.to_period("M")#把“调仓日”这一整列的每个日期，转换成对应的月份 Period
    == bt_signal_calendar["信号日"].dt.to_period("M") + 1
).all():
    raise AssertionError("部分月末信号未在下一自然月首个交易日调仓。")
if bt_signal_calendar["调仓日"].min() != BT_START_CLOSE:
    raise AssertionError("首个调仓日与设定的2016-02-01不一致。")

bt_constituent_weights = bt_constituent_weights.rename(
    columns={BT_WEIGHT_DATE: "信号日", BT_WEIGHT_CODE: BT_CODE}
).merge(#merge是合并表
    bt_signal_calendar,
    on="信号日",
    how="inner",
    validate="many_to_one",#左表的 "信号日" 可以重复很多次，右表的 "信号日" 必须唯一
)#不同于leftjoin，匹配不到左表会删除

bt_benchmark_codes = bt_constituent_weights[BT_CODE].drop_duplicates().tolist()#基准指数成分股权重表里曾经出现过的所有股票代码的去重列表
bt_industry_history = pd.read_parquet(
    BT_DAILY_PATH,
    columns=[BT_DATE, BT_CODE, BT_DAILY_INDUSTRY],#读取excel和csv都是usecols
    filters=[
        (BT_DATE, ">=", bt_signal_dates.min()),
        (BT_DATE, "<=", bt_signal_dates.max()),
        (BT_CODE, "in", bt_benchmark_codes),
    ],#(列名, 运算符, 值) 
)
bt_industry_history[BT_DATE] = pd.to_datetime(bt_industry_history[BT_DATE], errors="coerce")
bt_industry_history[BT_CODE] = bt_code6(bt_industry_history[BT_CODE])
bt_industry_history = bt_industry_history.loc[
    bt_industry_history[BT_DATE].isin(bt_signal_dates),
    [BT_DATE, BT_CODE, BT_DAILY_INDUSTRY],
].drop_duplicates([BT_DATE, BT_CODE])#默认保留第一条，keep=false两个都删掉

bt_constituent_weights = bt_constituent_weights.merge(
    bt_industry_history,
    left_on=["信号日", BT_CODE],
    right_on=[BT_DATE, BT_CODE],
    how="left",
    validate="one_to_one",
)#本质是leftjoin，只不过列名不同所以要left_on和right_on
if bt_constituent_weights[BT_DATE].isna().any():
    raise AssertionError("部分沪深300成分权重无法匹配信号日行情记录。")

bt_constituent_weights["行业一级代码"] = bt_level1_industry(
    bt_constituent_weights[BT_DAILY_INDUSTRY]
)#返回单个字符
bt_constituent_weights["行业一级"] = bt_constituent_weights[
    "行业一级代码"
].map(BT_CSRC_LEVEL1_NAMES)#匹配名字

bt_monthly_industry_weights = (
    bt_constituent_weights.groupby(
        ["信号日", "调仓日", "行业一级代码", "行业一级"],
        as_index=False,
        observed=True,
    )["个股权重"]
    .sum()
    .rename(columns={"个股权重": "行业权重"})
    .sort_values(["信号日", "行业一级代码"])
    .reset_index(drop=True)
)
bt_industry_weight_sums = bt_monthly_industry_weights.groupby("信号日")[
    "行业权重"
].sum()
if not np.allclose(bt_industry_weight_sums.to_numpy(), 1.0, atol=1e-12):#是否都接近 1.0
    raise AssertionError("部分月份的沪深300行业权重合计不等于1。")

bt_monthly_industry_weights.to_csv(
    BT_INDUSTRY_WEIGHT_PATH, index=False, encoding="utf-8-sig"
)

bt_signal_pool = bt_pool.loc[bt_pool[BT_DATE].isin(bt_signal_dates)].copy()
if BT_EXCLUDE_STAR:
    bt_signal_pool = bt_signal_pool.loc[
        ~bt_signal_pool[BT_CODE].str.startswith(("688", "689"), na=False)
    ].copy()
bt_signal_pool["行业一级代码"] = bt_level1_industry(
    bt_signal_pool[BT_POOL_INDUSTRY]
)
bt_signal_pool["行业一级"] = bt_signal_pool["行业一级代码"].map(
    BT_CSRC_LEVEL1_NAMES
)#根据字典返回对应的值 
 
bt_factor_data = pd.read_parquet(
    BT_FACTOR_PATH,
    columns=[BT_DATE, BT_CODE, *BT_FACTOR_COLUMNS.values()],
    filters=[
        (BT_DATE, ">=", bt_signal_dates.min()),
        (BT_DATE, "<=", bt_signal_dates.max()),
    ],
)
bt_factor_data[BT_DATE] = pd.to_datetime(bt_factor_data[BT_DATE], errors="coerce")
bt_factor_data[BT_CODE] = bt_code6(bt_factor_data[BT_CODE])
bt_factor_data = bt_factor_data.loc[bt_factor_data[BT_DATE].isin(bt_signal_dates)]#取回测里面的数据
if bt_factor_data.duplicated([BT_DATE, BT_CODE]).any():
    raise ValueError("中性化因子存在重复的日期-股票记录。")

bt_signal_pool = bt_signal_pool.merge(
    bt_factor_data,
    on=[BT_DATE, BT_CODE],
    how="left",
    validate="one_to_one",
).merge(
    bt_signal_calendar,
    left_on=[BT_DATE, BT_NEXT_DATE],
    right_on=["信号日", "调仓日"],
    how="inner",
    validate="many_to_one",
)#可以连续merge

bt_holding_parts = []#每个调仓期/信号日生成的持仓明细块
bt_allocation_rows = []#每次资产配置或行业分配的一行汇总结果
for bt_factor, bt_factor_column in BT_FACTOR_COLUMNS.items():
    for bt_signal_date, bt_targets in bt_monthly_industry_weights.groupby(
        "信号日", sort=True, observed=True
    ):
        bt_cross_section = bt_signal_pool.loc[
            bt_signal_pool["信号日"].eq(bt_signal_date)
        ]
        bt_rebalance_date = bt_targets["调仓日"].iloc[0]#第一个时间值

        for bt_industry_code, bt_industry_name, bt_industry_weight in bt_targets[
            ["行业一级代码", "行业一级", "行业权重"]
        ].itertuples(index=False, name=None):#按行遍历 DataFrame，不返回索引，不返回命名元组
            # 未知行业无法实施行业内选股，因此对应额度明确留作现金。
            if bt_industry_code == "UNKNOWN":
                bt_eligible = bt_cross_section.iloc[0:0].copy()#不读取为空
            else:
                bt_eligible = bt_cross_section.loc[
                    bt_cross_section["行业一级代码"].eq(bt_industry_code)#属于当前行业
                    & pd.to_numeric(
                        bt_cross_section[bt_factor_column], errors="coerce"#因子目标值不缺失
                    ).notna()
                ].copy()

            bt_eligible[bt_factor_column] = pd.to_numeric(
                bt_eligible[bt_factor_column], errors="coerce"
            )
            bt_eligible = bt_eligible.sort_values(
                [bt_factor_column, BT_CODE],
                ascending=[False, True],#降序，升序
                kind="mergesort",#归并排序
            )
            bt_selected_count = (
                int(math.ceil(len(bt_eligible) * BT_TOP_FRACTION))#ceil向上取整，看多少股票
                if len(bt_eligible)
                else 0
            )
            bt_selected = bt_eligible.head(bt_selected_count).copy()
            bt_cash_weight = float(bt_industry_weight) if bt_selected_count == 0 else 0.0#现金流，如果该行业不存在

            bt_allocation_rows.append({
                "因子": bt_factor,
                "信号日": bt_signal_date,
                "调仓日": bt_rebalance_date,
                "行业一级代码": bt_industry_code,
                "行业一级": bt_industry_name,
                "行业权重": float(bt_industry_weight),
                "行业有效候选数": len(bt_eligible),
                "行业入选数": bt_selected_count,
                "现金保留权重": bt_cash_weight,
            })

            if bt_selected_count == 0:
                continue

            bt_selected["因子"] = bt_factor
            bt_selected["因子值"] = bt_selected[bt_factor_column]
            bt_selected["目标权重"] = float(bt_industry_weight) / bt_selected_count
            bt_selected["行业有效候选数"] = len(bt_eligible)
            bt_selected["行业入选数"] = bt_selected_count
            bt_holding_parts.append(bt_selected[[
                "因子",
                "信号日",
                "调仓日",
                BT_CODE,
                "行业一级代码",
                "行业一级",
                "因子值",
                "行业有效候选数",
                "行业入选数",
                "目标权重",
            ]])

bt_monthly_holdings = pd.concat(bt_holding_parts, ignore_index=True)#拼接
bt_allocation_audit = pd.DataFrame(bt_allocation_rows)
bt_monthly_holdings = bt_monthly_holdings.sort_values(
    ["因子", "调仓日", "行业一级代码", "因子值", BT_CODE],
    ascending=[True, True, True, False, True],
).reset_index(drop=True)

bt_stock_weight_sums = bt_monthly_holdings.groupby(
    ["因子", "调仓日"], observed=True
)["目标权重"].sum()
bt_cash_weight_sums = bt_allocation_audit.groupby(
    ["因子", "调仓日"], observed=True
)["现金保留权重"].sum()
bt_allocation_check = pd.concat(
    [
        bt_stock_weight_sums.rename("股票目标权重"),
        bt_cash_weight_sums.rename("现金目标权重"),
    ],
    axis=1,#列方向拼接
).fillna(0.0)
bt_allocation_check["合计"] = (
    bt_allocation_check["股票目标权重"]
    + bt_allocation_check["现金目标权重"]
)
if not np.allclose(bt_allocation_check["合计"].to_numpy(), 1.0, atol=1e-12):#全部close
    raise AssertionError("股票与现金目标权重之和不等于1。")

bt_monthly_holdings.to_parquet(BT_HOLDINGS_PATH, index=False, compression="zstd")
bt_allocation_audit.to_csv(
    BT_ALLOCATION_AUDIT_PATH, index=False, encoding="utf-8-sig"
)

display(
    bt_monthly_industry_weights.pivot(#变换表的样式
        index="信号日",#行名
        columns="行业一级",#列名
        values="行业权重"#数值
    ).fillna(0.0).style.format("{:.2%}")#形式变为百分数
)
display(
    bt_monthly_holdings.groupby(["因子", "调仓日"], observed=True).agg(
        持仓数=(BT_CODE, "size"),#size是数量
        股票目标权重=("目标权重", "sum"),
    ).join(bt_cash_weight_sums.rename("现金目标权重")).head(12)#默认按照 index，把 B 的列横向接到 A 上,merge按照普通列连接
)

行业一级,交通运输、仓储和邮政业,信息传输、软件和信息技术服务业,农、林、牧、渔业,制造业,卫生和社会工作,建筑业,房地产业,批发和零售业,文化、体育和娱乐业,水利、环境和公共设施管理业,电力、热力、燃气及水生产和供应业,租赁和商务服务业,综合,行业缺失（权重保留现金）,采矿业,金融业
信号日,,,,,,,,,,,,,,,,
2016-01-29 00:00:00,3.43%,5.58%,0.08%,31.78%,0.15%,3.44%,5.77%,2.25%,1.58%,0.77%,3.96%,0.94%,0.46%,0.70%,3.59%,35.53%
2016-02-29 00:00:00,3.30%,5.40%,0.08%,31.92%,0.15%,3.42%,5.86%,2.28%,1.47%,0.74%,3.98%,0.91%,0.46%,0.70%,3.92%,35.43%
2016-03-31 00:00:00,3.27%,5.64%,0.08%,32.00%,0.14%,3.45%,5.41%,2.24%,1.61%,0.75%,3.92%,0.93%,0.49%,0.68%,3.68%,35.72%
2016-04-29 00:00:00,3.16%,5.52%,0.08%,31.72%,0.15%,3.33%,5.41%,2.21%,1.60%,0.75%,4.07%,0.91%,0.48%,0.69%,3.86%,36.04%
2016-05-31 00:00:00,3.19%,5.66%,0.08%,31.64%,0.15%,3.24%,5.35%,2.19%,1.61%,0.72%,4.03%,0.87%,0.45%,0.66%,3.67%,36.49%
2016-06-30 00:00:00,2.96%,6.36%,0.08%,31.31%,0.17%,3.07%,5.59%,2.10%,1.88%,0.70%,3.95%,1.21%,0.50%,0.64%,3.48%,36.00%
2016-07-29 00:00:00,3.14%,6.13%,0.08%,32.47%,0.17%,3.09%,4.93%,2.13%,1.77%,0.76%,4.01%,1.18%,0.49%,0.66%,3.61%,35.38%
2016-08-31 00:00:00,2.93%,6.17%,0.08%,31.45%,0.16%,3.26%,5.72%,2.12%,1.78%,0.78%,3.94%,1.18%,0.52%,0.70%,3.47%,35.76%
2016-09-30 00:00:00,2.94%,6.06%,0.08%,31.70%,0.16%,3.20%,5.89%,2.12%,1.73%,0.78%,3.90%,1.20%,0.50%,0.68%,3.41%,35.65%


持仓数    股票目标权重    现金目标权重
因子 调仓日                                
BP 2016-02-01  153  0.993000  0.007000
   2016-03-01  152  0.992969  0.007031
   2016-04-01  153  0.993180  0.006820
   2016-05-03  152  0.993050  0.006950
   2016-06-01  154  0.993380  0.006620
   2016-07-01  156  0.993580  0.006420
   2016-08-01  157  0.993370  0.006630
   2016-09-01  154  0.993040  0.006960
   2016-10-10  150  0.993210  0.006790
   2016-11-01  155  0.992940  0.007060
   2016-12-01  153  0.992930  0.007070
EP 2016-02-01  153  0.993000  0.007000

## 14. T+1 开盘换仓：停牌、一字涨跌停、订单重试与滑点

旧仓先获得前收盘至当日开盘的收益，随后按开盘价执行调仓，新仓再获得开盘至收盘的收益。停牌日不能买卖；最高价等于最低价且价格落在理论涨停价/跌停价时，分别视为一字涨停/一字跌停。尾盘涨停但盘中价格有变化时不属于一字板，允许在开盘成交。受阻订单保留到后续交易日重试；到回测截止日仍未完成的订单按期末到期取消，并另存逐笔原因，不做虚假强制成交。滑点按成交金额的固定比例扣减：买入价上浮、卖出价下浮。

In [16]:
def bt_round_half_up_to_cent(values):
    return np.floor(values * 100 + 0.5 + 1e-10) / 100#四舍五入


def bt_is_st_pt(values):
    status = values.astype("string").str.upper().str.strip()
    return status.str.contains(r"ST|PT", regex=True, na=False)#退市避雷


def bt_build_execution_flags(quotes):
    """用日线字段构造最简可交易状态。"""
    result = quotes.copy().sort_values([BT_CODE, BT_DATE]).reset_index(drop=True)#默认0
    result[BT_STATE] = result.groupby(BT_CODE, observed=True)[BT_STATE].ffill()

    numeric_columns = [
        BT_RETURN,
        BT_PREV_CLOSE,
        BT_OPEN,
        BT_HIGH,
        BT_LOW,
        BT_CLOSE,
        BT_VOLUME,
        BT_AMOUNT,
    ]
    result[numeric_columns] = result[numeric_columns].apply(
        pd.to_numeric, errors="coerce"
    )

    result["涨跌停板块"] = np.select(
        [
            result[BT_CODE].str.startswith(("300", "301")),
            result[BT_CODE].str.startswith(("688", "689")),
        ],
        ["CHINEXT", "STAR"],
        default="MAIN",
    )
    unsupported = result[BT_CODE].str.startswith(("4", "8", "9"), na=False)
    if unsupported.any():
        bad_codes = result.loc[unsupported, BT_CODE].drop_duplicates().head(10).tolist()
        raise ValueError(f"存在未配置涨跌停规则的证券代码：{bad_codes}")

    trade_day_number = pd.Series(
        np.arange(len(all_trading_dates)), index=all_trading_dates
    )
    result["上市日期近似"] = result[BT_CODE].map(first_seen_date)
    result["_上市日序号"] = result["上市日期近似"].map(trade_day_number)
    result["_当日序号"] = result[BT_DATE].map(trade_day_number)
    result["_数据期内新股"] = result["上市日期近似"].gt(dataset_start)
    result["_无涨跌停天数"] = np.select(
        [
            result["涨跌停板块"].eq("STAR"),
            result["涨跌停板块"].eq("CHINEXT")
            & result["上市日期近似"].ge(pd.Timestamp("2020-08-24")),
            result["涨跌停板块"].eq("MAIN")
            & result["上市日期近似"].ge(pd.Timestamp("2023-04-10")),
        ],
        [5, 5, 5],#5天不设置涨跌幅
        default=1,
    )
    result["是否无涨跌停限制期"] = (
        result["_数据期内新股"]
        & (
            result["_当日序号"] - result["_上市日序号"]
            < result["_无涨跌停天数"]
        )
    )

    result["是否ST_PT"] = bt_is_st_pt(result[BT_STATE])
    result["涨跌停幅度"] = np.select(
        [
            result["是否ST_PT"],
            result["涨跌停板块"].eq("STAR"),
            result["涨跌停板块"].eq("CHINEXT")
            & result[BT_DATE].ge(pd.Timestamp("2020-08-24")),
        ],
        [0.05, 0.20, 0.20],
        default=0.10,
    )
    result["理论涨停价"] = bt_round_half_up_to_cent(
        result[BT_PREV_CLOSE] * (1 + result["涨跌停幅度"])
    )
    result["理论跌停价"] = bt_round_half_up_to_cent(
        result[BT_PREV_CLOSE] * (1 - result["涨跌停幅度"])
    )

    result["是否停牌"] = (
        result[[BT_OPEN, BT_HIGH, BT_LOW, BT_CLOSE]].isna().any(axis=1)
        | result[BT_VOLUME].isna()
        | result[BT_AMOUNT].isna()
        | result[BT_VOLUME].le(0)
        | result[BT_AMOUNT].le(0)
    )
    result["是否一字"] = (
        ~result["是否停牌"]
        & np.isclose(result[BT_HIGH], result[BT_LOW], atol=0.00501, rtol=0)#看最高价和最低价
    )
    result["是否一字涨停"] = (
        result["是否一字"]
        & ~result["是否无涨跌停限制期"]
        & np.isclose(
            result[BT_HIGH], result["理论涨停价"], atol=0.00501, rtol=0
        )
    )
    result["是否一字跌停"] = (
        result["是否一字"]
        & ~result["是否无涨跌停限制期"]
        & np.isclose(
            result[BT_LOW], result["理论跌停价"], atol=0.00501, rtol=0
        )
    )
    result["是否收盘涨停"] = (
        ~result["是否停牌"]
        & ~result["是否无涨跌停限制期"]
        & np.isclose(
            result[BT_CLOSE], result["理论涨停价"], atol=0.00501, rtol=0
        )
    )
    result["是否尾盘涨停非一字"] = (
        result["是否收盘涨停"] & ~result["是否一字涨停"]
    )
    result["可买入"] = ~result["是否停牌"] & ~result["是否一字涨停"]
    result["可卖出"] = ~result["是否停牌"] & ~result["是否一字跌停"]

    traded = ~result["是否停牌"]
    if result.loc[traded, BT_RETURN].isna().any():
        bad = result.loc[traded & result[BT_RETURN].isna(), [BT_DATE, BT_CODE]].head()
        raise ValueError(f"存在正常成交但日收益率缺失的记录：\n{bad}")
    result["开盘至收盘收益率"] = np.where(
        traded,
        result[BT_CLOSE] / result[BT_OPEN] - 1.0,
        0.0,
    )#np.where(条件, 条件为True时的值, 条件为False时的值)
    result["前收盘至开盘收益率"] = np.where(
        traded,
        (1.0 + result[BT_RETURN])
        / (1.0 + result["开盘至收盘收益率"])
        - 1.0,
        0.0,
    )#跳空开盘收益率
    reconstructed_return = (
        (1.0 + result["前收盘至开盘收益率"])  
        * (1.0 + result["开盘至收盘收益率"])
        - 1.0
    )
    if not np.allclose(
        reconstructed_return[traded],
        result.loc[traded, BT_RETURN],
        rtol=0,
        atol=1e-12,
    ):
        raise AssertionError("隔夜收益与日内收益未能还原日收益率。")
    return result


def bt_make_target_schedule(frame, group_column, weight_column):#当前这个因子的持仓表， BT_CODE，也就是股票代码列，"目标权重"
    schedules = {}
    for rebalance_date, group in frame.groupby("调仓日", sort=True, observed=True):
          #weights = group.groupby(group_column, observed=True)[weight_column].sum()#保守措施
          weights = group.set_index(group_column)[weight_column]
          schedules[pd.Timestamp(rebalance_date)] = weights[weights.gt(0)].astype(float)
    return schedules


def bt_execute_pending_orders(
    positions,              # 当前持仓：通常记录每只股票当前持有的市值/股数
    cash,                   # 当前可用现金
    target_weights,         # 目标权重：每只股票最终希望调整到的组合权重
    pending_codes,          # 待执行订单的股票代码集合
    can_buy_today,          # 今日是否允许买入：按股票代码记录 True/False
    can_sell_today,         # 今日是否允许卖出：按股票代码记录 True/False
    suspended_today,        # 今日是否停牌：按股票代码记录 True/False
    one_price_up_today,     # 今日是否一字涨停：一字涨停通常无法买入
    one_price_down_today,   # 今日是否一字跌停：一字跌停通常无法卖出
    fee_rate,               # 单边交易手续费率
    slippage_rate,          # 滑点率：模拟实际成交价相对理论价格的偏差
):  
    target_weight_sum = float(target_weights.sum())
    
    if target_weight_sum > 1.0 + 1e-12:
            raise AssertionError(
                f"目标权重和超过1：{target_weight_sum:.8f}"
            )
    
    if (target_weights < -1e-12).any():
            raise AssertionError("目标权重存在负值。")
    """先卖后买；不能成交的代码保留到下一交易日继续尝试。"""
    pre_trade_nav = float(positions.sum() + cash)
    universe = positions.index.union(target_weights.index).union(
        pd.Index(sorted(pending_codes), dtype="object")
    )#union是取并集，把目标和待买和持仓的合起来了
    current_values = positions.reindex(universe, fill_value=0.0).astype(float)
    weights = target_weights.reindex(universe, fill_value=0.0).astype(float)
    desired_values = weights * pre_trade_nav
    pending_mask = pd.Series(universe.isin(pending_codes), index=universe)#index的isin返回的是bool数组
    tolerance = max(1e-12, pre_trade_nav * 1e-10)

    initial_gap = (desired_values - current_values).where(pending_mask, 0.0)
    sell_request = (-initial_gap.clip(upper=0.0)).clip(upper=current_values)#第二个clip表示卖出金额不能超过当前金额
    sellable = can_sell_today.reindex(universe).fillna(False).astype(bool)#空的先fasle，防御
    sell_execution = sell_request.where(sellable, 0.0)#执行

    sell_notional = float(sell_execution.sum())
    blocked_sell_notional = float((sell_request - sell_execution).sum())#没卖出去的
    positions = current_values - sell_execution#更新
    sell_fee = sell_notional * fee_rate#手续费
    sell_slippage = sell_notional * slippage_rate#滑点
    cash = float(cash + sell_notional - sell_fee - sell_slippage)#名义现金

    after_sell_values = positions.reindex(universe, fill_value=0.0)
    buy_request = (desired_values - after_sell_values).clip(lower=0.0).where(
        pending_mask, 0.0#不属于则为0
    )#想买的
    buyable = can_buy_today.reindex(universe).fillna(False).astype(bool)
    eligible_buy_request = buy_request.where(buyable, 0.0)#不能买记住0

    target_cash_value = (
        1.0 - target_weight_sum
    ) * pre_trade_nav#目标组合应该保留多少现金
    spendable_cash = max(0.0, cash - target_cash_value)#可以花的现金
    gross_cost_per_buy = 1.0 + fee_rate + slippage_rate
    affordable_buy = spendable_cash / gross_cost_per_buy#能买的
    eligible_buy_total = float(eligible_buy_request.sum())
    buy_scale = (
        min(1.0, affordable_buy / eligible_buy_total)
        if eligible_buy_total > 0
        else 0.0
    )
    buy_execution = eligible_buy_request * buy_scale#钱不够则执行缩放
    # 如果今天所有应该卖出的股票都成功卖出，
    # 本来还能释放多少净现金
    cash_if_all_sells_executed = (
        cash
        + blocked_sell_notional * (1.0 - fee_rate - slippage_rate)
    )
    # 在“没有卖单受阻”的假设下，可以真正用于买入的现金
    spendable_cash_without_block = max(
        0.0,
        cash_if_all_sells_executed - target_cash_value
    )
    # 没有卖单受阻时，最多能买多少股票
    affordable_buy_without_block = (
        spendable_cash_without_block / gross_cost_per_buy
    )
    # 没有卖单受阻时，买单理论上的缩放比例
    buy_scale_without_block = (
        min(1.0, affordable_buy_without_block / eligible_buy_total)
        if eligible_buy_total > 0
        else 0.0
    )
    # 如果没有卖单受阻，各股票今天理论上应该成交多少
    buy_execution_without_block = (
        eligible_buy_request * buy_scale_without_block
    )
    buy_notional = float(buy_execution.sum())
    buy_fee = buy_notional * fee_rate
    buy_slippage = buy_notional * slippage_rate
    positions = positions + buy_execution
    cash = float(cash - buy_notional - buy_fee - buy_slippage)
    if cash < 0 and abs(cash) <= tolerance:
        cash = 0.0

    if cash < -tolerance:
        raise AssertionError(
            "交易后现金为负，请检查手续费和滑点处理。"
        )

    positions = positions[positions.abs().gt(1e-15)]
    remaining_codes = set()

    for stock_code in universe[pending_mask.to_numpy()]:

        gap = float(initial_gap.loc[stock_code])

        if abs(gap) <= tolerance:
            continue

        # --------------------
        # 卖单
        # --------------------
        if gap < 0:

            requested = float(sell_request.loc[stock_code])
            executed = float(sell_execution.loc[stock_code])

            if (
                not bool(sellable.loc[stock_code])
                or executed + tolerance < requested
            ):
                remaining_codes.add(str(stock_code))

        # --------------------
        # 买单
        # --------------------
        else:
            executed = float(buy_execution.loc[stock_code])
            # 今天根本买不了
            if not bool(buyable.loc[stock_code]):
                remaining_codes.add(str(stock_code))
                continue

            # 假设卖单全部成功，本来应该买多少
            executable_without_block = float(
                buy_execution_without_block.loc[stock_code]
            )

            # 实际成交比“无卖单受阻情况下”还少，
            # 说明少买的部分确实是被卖单受阻卡住了
            if executed + tolerance < executable_without_block:
                remaining_codes.add(str(stock_code))

    suspended = suspended_today.reindex(universe).fillna(True).astype(bool)
    one_up = one_price_up_today.reindex(universe).fillna(False).astype(bool)
    one_down = one_price_down_today.reindex(universe).fillna(False).astype(bool)
    blocked_buy = buy_request.gt(tolerance) & ~buyable
    blocked_sell = sell_request.gt(tolerance) & ~sellable

    pending_detail_columns = [
        BT_CODE, "待成交方向", "待成交原因", "申请金额", "已成交金额",
        "未成交金额", "是否停牌", "是否一字涨停", "是否一字跌停",
        "当日可买入", "当日可卖出",
    ]
    pending_detail_rows = []
    for stock_code in sorted(remaining_codes):
        gap = float(initial_gap.loc[stock_code])
        if gap < 0:
            direction = "卖出"
            requested = float(sell_request.loc[stock_code])
            executed = float(sell_execution.loc[stock_code])
            if bool(suspended.loc[stock_code]):
                reason = "停牌导致卖出受阻"
            elif bool(one_down.loc[stock_code]):
                reason = "一字跌停导致卖出受阻"
            else:
                reason = "其他原因导致卖出未完成"
        else:
            direction = "买入"
            requested = float(buy_request.loc[stock_code])
            executed = float(buy_execution.loc[stock_code])
            if bool(suspended.loc[stock_code]):
                reason = "停牌导致买入受阻"
            elif bool(one_up.loc[stock_code]):
                reason = "一字涨停导致买入受阻"
            elif not bool(buyable.loc[stock_code]):
                reason = "其他不可买入状态"
            else:
                reason = "卖出受阻导致可用现金不足"

        pending_detail_rows.append({
            BT_CODE: str(stock_code),
            "待成交方向": direction,
            "待成交原因": reason,
            "申请金额": requested,
            "已成交金额": executed,
            "未成交金额": max(0.0, requested - executed),
            "是否停牌": bool(suspended.loc[stock_code]),
            "是否一字涨停": bool(one_up.loc[stock_code]),
            "是否一字跌停": bool(one_down.loc[stock_code]),
            "当日可买入": bool(buyable.loc[stock_code]),
            "当日可卖出": bool(sellable.loc[stock_code]),
        })
    pending_detail = pd.DataFrame(
        pending_detail_rows, columns=pending_detail_columns
    )

    stats = {
        "申请买入金额": float(buy_request.sum()),
        "申请卖出金额": float(sell_request.sum()),
        "买入金额": buy_notional,
        "卖出金额": sell_notional,
        "手续费": float(buy_fee + sell_fee),
        "滑点成本": float(buy_slippage + sell_slippage),
        "现金受限未买金额": float((eligible_buy_request - buy_execution).sum()),
        "受阻买入股票数": int(blocked_buy.sum()),
        "受阻卖出股票数": int(blocked_sell.sum()),
        "停牌受阻买入数": int((blocked_buy & suspended).sum()),
        "停牌受阻卖出数": int((blocked_sell & suspended).sum()),
        "一字涨停受阻买入数": int((blocked_buy & one_up).sum()),
        "一字跌停受阻卖出数": int((blocked_sell & one_down).sum()),
    }
    return positions, cash, remaining_codes, stats, pending_detail


def bt_simulate_open_rebalance(
    name,                     # 回测策略/因子名称，用于结果标识
    dates,                    # 回测交易日期序列
    full_day_return_panel,    # 股票全天收益率面板：通常为前一日收盘 → 当日收盘收益率
    overnight_return_panel,   # 股票隔夜收益率面板：通常为前一日收盘 → 当日开盘收益率
    intraday_return_panel,    # 股票日内收益率面板：通常为当日开盘 → 当日收盘收益率
    target_schedule,          # 目标持仓权重计划：记录各调仓日对应的目标股票及目标权重
    can_buy_panel,            # 是否允许买入的布尔面板：True 表示当天开盘可买
    can_sell_panel,           # 是否允许卖出的布尔面板：True 表示当天开盘可卖
    suspended_panel,          # 停牌状态面板：True 表示当天股票停牌
    one_price_up_panel,       # 一字涨停状态面板：True 表示当天一字涨停，通常无法买入
    one_price_down_panel,     # 一字跌停状态面板：True 表示当天一字跌停，通常无法卖出
    fee_rate,                 # 单边交易手续费率
    slippage_rate             # 单边交易滑点率，用于模拟实际成交价格偏离理论开盘价
):
    positions = pd.Series(dtype=float)
    cash = 1.0
    last_nav = 1.0
    current_target = pd.Series(dtype=float)#空float
    current_target_date = pd.NaT#no time
    pending_codes = set()
    pending_detail = pd.DataFrame()
    daily_rows = []
    trade_rows = []

    for trade_date in dates:
        missing_return_count = 0
        suspended_missing_count = 0
        abnormal_missing_count = 0
        if not positions.empty:
            full_day_returns = full_day_return_panel.loc[trade_date].reindex(
                positions.index
            )

            suspended_today = suspended_panel.loc[trade_date].reindex(
                positions.index
            ).fillna(True)

            missing_mask = full_day_returns.isna()

            # 所有收益率缺失
            missing_return_count = int(missing_mask.sum())

            # 收益缺失，同时当天停牌
            suspended_missing_count = int(
                (missing_mask & suspended_today).sum()
            )

            # 收益缺失，但当天并没有停牌
            abnormal_missing_count = int(
                (missing_mask & ~suspended_today).sum()
            )

            overnight_returns = overnight_return_panel.loc[trade_date].reindex(
                positions.index
            )

            positions = positions * (
                1.0 + overnight_returns.fillna(0.0)
            )

        is_new_rebalance = trade_date in target_schedule#返回的是一个布尔值 True 或 False
        if is_new_rebalance:
            current_target = target_schedule[trade_date]
            if current_target.sum() > 1.0 + 1e-12:
                raise AssertionError(f"{name}在{trade_date.date()}的目标权重超过1。")
            current_target_date = trade_date
            pending_codes = set(positions.index).union(current_target.index)

        pre_trade_nav = float(positions.sum() + cash)#全部净值
        stats = {
            "申请买入金额": 0.0,
            "申请卖出金额": 0.0,
            "买入金额": 0.0,
            "卖出金额": 0.0,
            "手续费": 0.0,
            "滑点成本": 0.0,
            "现金受限未买金额": 0.0,
            "受阻买入股票数": 0,
            "受阻卖出股票数": 0,
            "停牌受阻买入数": 0,
            "停牌受阻卖出数": 0,
            "一字涨停受阻买入数": 0,
            "一字跌停受阻卖出数": 0,
        }#初始

        had_pending_orders = bool(pending_codes)
        if had_pending_orders:
            positions, cash, pending_codes, stats, pending_detail = (
                bt_execute_pending_orders(
                positions=positions,
                cash=cash,
                target_weights=current_target,
                pending_codes=pending_codes,
                can_buy_today=can_buy_panel.loc[trade_date],
                can_sell_today=can_sell_panel.loc[trade_date],
                suspended_today=suspended_panel.loc[trade_date],
                one_price_up_today=one_price_up_panel.loc[trade_date],
                one_price_down_today=one_price_down_panel.loc[trade_date],
                fee_rate=fee_rate,
                slippage_rate=slippage_rate,
                )
            )#先执行待执行订单，每天都来一次

        post_trade_open_nav = float(positions.sum() + cash)#理论
        expected_open_nav = pre_trade_nav - stats["手续费"] - stats["滑点成本"]#期望
        if not np.isclose(post_trade_open_nav, expected_open_nav, rtol=0, atol=1e-12):
            raise AssertionError(f"{name}在{trade_date.date()}开盘交易后的资产未配平。")

        if not positions.empty:#推进到今天收盘
            intraday_returns = intraday_return_panel.loc[trade_date].reindex(
                positions.index
            )
            positions = positions * (1.0 + intraday_returns.fillna(0.0))
        nav = float(positions.sum() + cash)#收盘净值

        suspended_held = suspended_panel.loc[trade_date].reindex(
            positions.index
        ).fillna(True)#看看position中有无停牌的
        one_down_held = one_price_down_panel.loc[trade_date].reindex(
            positions.index
        ).fillna(False)#看持仓一字跌停的
        suspended_value = float(positions[suspended_held].sum())#停牌
        one_down_value = float(positions[one_down_held].sum())#一字跌停

        if is_new_rebalance or had_pending_orders or stats["买入金额"] > 0 or stats["卖出金额"] > 0:#发生交易或者大换仓或者有未执行订单
            trade_rows.append({
                "组合": name,
                "交易日": trade_date,
                "目标调仓日": current_target_date,
                "是否新调仓": is_new_rebalance,
                "开盘交易前净值": pre_trade_nav,
                "开盘交易后净值": post_trade_open_nav,
                **stats,
                "买入换手率": stats["买入金额"] / pre_trade_nav,
                "卖出换手率": stats["卖出金额"] / pre_trade_nav,
                "双边换手率": (
                    stats["买入金额"] + stats["卖出金额"]
                ) / pre_trade_nav,
                "总交易成本": stats["手续费"] + stats["滑点成本"],
                "持仓数": len(positions),
                "目标现金权重": 1.0 - float(current_target.sum()),
                "未完成交易股票数": len(pending_codes),
            })

        daily_rows.append({
            "组合": name,
            BT_DATE: trade_date,
            "日收益率": nav / last_nav - 1.0,
            "净值": nav,
            "现金权重": cash / nav if nav > 0 else np.nan,
            "持仓收益缺失数": missing_return_count,
            "停牌导致收益缺失数": suspended_missing_count,
            "非停牌收益缺失数": abnormal_missing_count,
            "当日手续费": stats["手续费"],
            "当日滑点成本": stats["滑点成本"],
            "停牌持仓市值": suspended_value,
            "一字跌停持仓市值": one_down_value,
            "受阻买入股票数": stats["受阻买入股票数"],
            "受阻卖出股票数": stats["受阻卖出股票数"],
            "停牌受阻卖出数": stats["停牌受阻卖出数"],
            "一字跌停受阻卖出数": stats["一字跌停受阻卖出数"],
            "未完成交易股票数": len(pending_codes),
        })
        last_nav = nav
        

    final_pending_columns = [
        "组合", "回测截止日", "目标调仓日", BT_CODE, "待成交方向",
        "待成交原因", "申请金额", "已成交金额", "未成交金额",
        "是否停牌", "是否一字涨停", "是否一字跌停",
        "当日可买入", "当日可卖出", "订单状态",
    ]
    if not pending_detail.empty:
        pending_detail = pending_detail.assign(
            组合=name,
            回测截止日=pd.Timestamp(dates[-1]),
            目标调仓日=current_target_date,
            订单状态="回测期末到期取消",
        )[final_pending_columns]
    else:
        pending_detail = pd.DataFrame(columns=final_pending_columns)

    return (
        pd.DataFrame(daily_rows),
        pd.DataFrame(trade_rows),
        pending_detail,
    )


bt_factor_schedules = {
    factor: bt_make_target_schedule(
        bt_monthly_holdings.loc[bt_monthly_holdings["因子"].eq(factor)],
        BT_CODE,
        "目标权重",
    )
    for factor in BT_FACTORS
}#字典
bt_trading_dates = pd.DatetimeIndex(
    pd.concat([bt_pool[BT_DATE], bt_pool[BT_NEXT_DATE]])
    .dropna()
    .drop_duplicates()
    .sort_values()
)
bt_backtest_dates = bt_trading_dates[
    (bt_trading_dates >= BT_START_CLOSE) & (bt_trading_dates <= BT_END)
]
bt_required_codes = sorted(set(bt_monthly_holdings[BT_CODE].dropna()))

bt_quote_columns = [
    BT_DATE,
    BT_CODE,
    BT_RETURN,
    BT_STATE,
    BT_PREV_CLOSE,
    BT_OPEN,
    BT_HIGH,
    BT_LOW,
    BT_CLOSE,
    BT_VOLUME,
    BT_AMOUNT,
]
bt_daily_quotes = pd.read_parquet(
    BT_DAILY_PATH,
    columns=bt_quote_columns,
    filters=[
        (BT_DATE, ">=", BT_START_CLOSE),
        (BT_DATE, "<=", BT_END),
        (BT_CODE, "in", bt_required_codes),
    ],
)
bt_daily_quotes[BT_DATE] = pd.to_datetime(
    bt_daily_quotes[BT_DATE], errors="coerce"
)
bt_daily_quotes[BT_CODE] = bt_code6(bt_daily_quotes[BT_CODE])
if bt_daily_quotes.duplicated([BT_DATE, BT_CODE]).any():
    raise ValueError("回测行情存在重复的日期-股票记录。")

bt_tradeability = bt_build_execution_flags(bt_daily_quotes)#生成那些限制列
bt_tradeability.to_parquet(BT_TRADEABILITY_PATH, index=False, compression="zstd")

def bt_value_panel(value_column):
    return bt_tradeability.pivot(
        index=BT_DATE, columns=BT_CODE, values=value_column
    ).reindex(index=bt_backtest_dates, columns=bt_required_codes)#


bt_return_panel = bt_value_panel(BT_RETURN)
bt_overnight_return_panel = bt_value_panel("前收盘至开盘收益率")
bt_intraday_return_panel = bt_value_panel("开盘至收盘收益率")
bt_can_buy_panel = bt_value_panel("可买入").fillna(False).astype(bool)
bt_can_sell_panel = bt_value_panel("可卖出").fillna(False).astype(bool)
bt_suspended_panel = bt_value_panel("是否停牌").fillna(True).astype(bool)
bt_one_price_up_panel = bt_value_panel("是否一字涨停").fillna(False).astype(bool)
bt_one_price_down_panel = bt_value_panel("是否一字跌停").fillna(False).astype(bool)

bt_daily_parts = []
bt_trade_parts = []
bt_end_pending_parts = []
for bt_factor in BT_FACTORS:
    bt_factor_daily, bt_factor_trades, bt_factor_end_pending = (
        bt_simulate_open_rebalance(
        name=bt_factor,
        dates=bt_backtest_dates,
        full_day_return_panel=bt_return_panel,
        overnight_return_panel=bt_overnight_return_panel,
        intraday_return_panel=bt_intraday_return_panel,
        target_schedule=bt_factor_schedules[bt_factor],
        can_buy_panel=bt_can_buy_panel,
        can_sell_panel=bt_can_sell_panel,
        suspended_panel=bt_suspended_panel,
        one_price_up_panel=bt_one_price_up_panel,
        one_price_down_panel=bt_one_price_down_panel,
        fee_rate=BT_ONE_SIDE_FEE_RATE,
        slippage_rate=BT_SLIPPAGE_RATE,
        )
    )
    bt_daily_parts.append(bt_factor_daily)
    bt_trade_parts.append(bt_factor_trades)
    bt_end_pending_parts.append(bt_factor_end_pending)

BT_BENCHMARK_NAME = "沪深300指数"

bt_index_quotes = pd.read_parquet(
    BT_INDEX_PATH,
    columns=[
        BT_INDEX_CODE,
        BT_INDEX_DATE,
        BT_INDEX_RETURN,
        BT_INDEX_OPEN,
        BT_INDEX_CLOSE,
    ],
    filters=[
        (BT_INDEX_DATE, ">=", BT_START_CLOSE),
        (BT_INDEX_DATE, "<=", BT_END),
    ],
)
bt_index_quotes[BT_INDEX_CODE] = bt_code6(bt_index_quotes[BT_INDEX_CODE])
bt_index_quotes[BT_INDEX_DATE] = pd.to_datetime(
    bt_index_quotes[BT_INDEX_DATE], errors="coerce"
)
bt_index_quotes[BT_INDEX_RETURN] = pd.to_numeric(
    bt_index_quotes[BT_INDEX_RETURN], errors="coerce"
)
bt_index_quotes[BT_INDEX_OPEN] = pd.to_numeric(
    bt_index_quotes[BT_INDEX_OPEN], errors="coerce"
)
bt_index_quotes[BT_INDEX_CLOSE] = pd.to_numeric(
    bt_index_quotes[BT_INDEX_CLOSE], errors="coerce"
)

bt_hs300 = bt_index_quotes.loc[
    bt_index_quotes[BT_INDEX_CODE].eq(BT_HS300_CODE)
].copy()
if bt_hs300.duplicated(BT_INDEX_DATE).any():
    raise ValueError("沪深300指数行情存在重复交易日。")

bt_hs300_returns = (
    bt_hs300.set_index(BT_INDEX_DATE)[BT_INDEX_RETURN]
    .reindex(bt_backtest_dates)
    .copy()
)
if bt_hs300_returns.isna().any():
    missing_dates = bt_hs300_returns.index[bt_hs300_returns.isna()].tolist()
    raise ValueError(f"沪深300指数行情未覆盖全部回测日：{missing_dates[:10]}")

# 策略在首个交易日开盘建仓，基准首日也只计算开盘至收盘收益。
bt_hs300_first_intraday_return = float(
    bt_hs300.set_index(BT_INDEX_DATE).loc[bt_backtest_dates[0], BT_INDEX_CLOSE]
    / bt_hs300.set_index(BT_INDEX_DATE).loc[bt_backtest_dates[0], BT_INDEX_OPEN]- 1.0
)
bt_hs300_returns.iloc[0] = bt_hs300_first_intraday_return
bt_hs300_nav = (1.0 + bt_hs300_returns).cumprod()
bt_benchmark_daily = pd.DataFrame({
    "组合": BT_BENCHMARK_NAME,
    BT_DATE: bt_backtest_dates,
    "日收益率": bt_hs300_returns.to_numpy(),
    "净值": bt_hs300_nav.to_numpy(),
    "现金权重": 0.0,
    "持仓收益缺失数": 0,
    "当日手续费": 0.0,
    "当日滑点成本": 0.0,
    "停牌持仓市值": 0.0,
    "一字跌停持仓市值": 0.0,
    "受阻买入股票数": 0,
    "受阻卖出股票数": 0,
    "停牌受阻卖出数": 0,
    "一字跌停受阻卖出数": 0,
    "未完成交易股票数": 0,
})
bt_daily_parts.append(bt_benchmark_daily)

bt_daily_backtest = pd.concat(bt_daily_parts, ignore_index=True)
bt_turnover_detail = pd.concat(bt_trade_parts, ignore_index=True)
bt_end_pending_detail = pd.concat(
    bt_end_pending_parts, ignore_index=True
)
bt_reported_end_pending = int(
    bt_daily_backtest.loc[
        bt_daily_backtest["组合"].isin(BT_FACTORS)
    ].sort_values(BT_DATE).groupby("组合").tail(1)["未完成交易股票数"].sum()
)
if bt_end_pending_detail.duplicated(["组合", BT_CODE]).any():
    raise AssertionError("期末待成交明细存在重复的组合-股票记录。")
if len(bt_end_pending_detail) != bt_reported_end_pending:
    raise AssertionError("期末待成交明细与日度汇总数量不一致。")
bt_daily_backtest.to_parquet(
    BT_DAILY_RESULT_PATH, index=False, compression="zstd"
)
bt_turnover_detail.to_parquet(
    BT_TURNOVER_PATH, index=False, compression="zstd"
)
bt_end_pending_detail.to_csv(
    BT_END_PENDING_PATH, index=False, encoding="utf-8-sig"
)

display(
    bt_turnover_detail.groupby("组合", observed=True).agg(
        目标调仓次数=("是否新调仓", "sum"),
        交易尝试天数=("交易日", "nunique"),
        累计买入换手率=("买入换手率", "sum"),
        累计卖出换手率=("卖出换手率", "sum"),
        累计手续费=("手续费", "sum"),
        累计滑点成本=("滑点成本", "sum"),
        停牌受阻卖出次数=("停牌受阻卖出数", "sum"),
        一字跌停受阻卖出次数=("一字跌停受阻卖出数", "sum"),
    )
)

C:\Temp\ipykernel_42532\2877892004.py:633: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bt_can_buy_panel = bt_value_panel("可买入").fillna(False).astype(bool)
C:\Temp\ipykernel_42532\2877892004.py:634: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bt_can_sell_panel = bt_value_panel("可卖出").fillna(False).astype(bool)
C:\Temp\ipykernel_42532\2877892004.py:635: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavi

,目标调仓次数,交易尝试天数,累计买入换手率,累计卖出换手率,累计手续费,累计滑点成本,停牌受阻卖出次数,一字跌停受阻卖出次数
组合,,,,,,,,
BP,11,209,1.827383,0.847735,0.011639,0.001455,691,0
EP,11,209,2.512790,1.539507,0.018369,0.002296,526,0
SP,11,209,1.988342,1.010171,0.013209,0.001651,620,0


## 15. 计算绩效指标并输出检查结果

年化按252个交易日；夏普使用 `RESSET_BDDRFRET_1.xlsx` 中已经换算为小数形式的日无风险收益率，并按日期与策略日收益严格对齐后直接相减；信息比率使用策略日收益减沪深300指数日收益。最大回撤从初始净值1开始计算，因此包含初始建仓费用的影响。

In [17]:
def bt_max_drawdown(nav):
    values = pd.to_numeric(nav, errors="coerce").dropna().to_numpy(float)
    values = np.r_[1.0, values]#把1放在前面
    running_max = np.maximum.accumulate(values)#运算从左往右不断累计
    return float(np.min(values / running_max - 1.0))#最大回撤


bt_nav_table = bt_daily_backtest.pivot(
    index=BT_DATE, columns="组合", values="净值"
).sort_index()
bt_return_table = bt_daily_backtest.pivot(
    index=BT_DATE, columns="组合", values="日收益率"
).sort_index()

# RESSET文件的工作表范围声明异常，必须先重置范围才能读取完整两列数据。
bt_risk_free_workbook = load_workbook(
    BT_RISK_FREE_PATH, read_only=True, data_only=True
)
try:
    bt_risk_free_worksheet = bt_risk_free_workbook["BDDRFRET"]
    bt_risk_free_worksheet.reset_dimensions()
    bt_risk_free_rows = bt_risk_free_worksheet.iter_rows(values_only=True)
    bt_risk_free_columns = list(next(bt_risk_free_rows))
    bt_risk_free_data = pd.DataFrame(
        bt_risk_free_rows, columns=bt_risk_free_columns
    )
finally:
    bt_risk_free_workbook.close()

bt_required_rf_columns = {BT_RF_DATE, BT_RF_RETURN}
if not bt_required_rf_columns.issubset(bt_risk_free_data.columns):
    raise KeyError(
        f"无风险收益率文件缺少字段："
        f"{sorted(bt_required_rf_columns - set(bt_risk_free_data.columns))}"
    )

bt_risk_free_data = bt_risk_free_data[[BT_RF_DATE, BT_RF_RETURN]].copy()
bt_risk_free_data[BT_RF_DATE] = pd.to_datetime(
    bt_risk_free_data[BT_RF_DATE], errors="coerce"
)
bt_risk_free_data[BT_RF_RETURN] = pd.to_numeric(
    bt_risk_free_data[BT_RF_RETURN], errors="coerce"
)
if bt_risk_free_data[[BT_RF_DATE, BT_RF_RETURN]].isna().any().any():
    raise ValueError("无风险收益率文件存在无法解析的日期或收益率")
if bt_risk_free_data[BT_RF_DATE].duplicated().any():
    raise ValueError("无风险收益率文件存在重复日期")

bt_risk_free_daily = (
    bt_risk_free_data.set_index(BT_RF_DATE)[BT_RF_RETURN]
    .sort_index()
    .reindex(bt_return_table.index)
)
bt_risk_free_daily.name = BT_RF_RETURN
if bt_risk_free_daily.isna().any():
    bt_missing_rf_dates = bt_risk_free_daily.index[
        bt_risk_free_daily.isna()
    ].strftime("%Y-%m-%d").tolist()
    raise ValueError(
        f"回测交易日缺少日无风险收益率，共{len(bt_missing_rf_dates)}天："
        f"{bt_missing_rf_dates[:5]}"
    )
if bt_risk_free_daily.abs().max() >= 0.01:
    raise ValueError(
        "日无风险收益率绝对值达到1%，请检查是否误把百分数当成小数"
    )
bt_benchmark_returns = bt_return_table[BT_BENCHMARK_NAME]
bt_benchmark_nav = bt_nav_table[BT_BENCHMARK_NAME]
bt_observations = len(bt_nav_table)

bt_performance_rows = []
for bt_factor in BT_FACTORS:
    bt_strategy_returns = bt_return_table[bt_factor]#每天收益
    bt_strategy_nav = bt_nav_table[bt_factor]#每天净值
    bt_active_returns = bt_strategy_returns - bt_benchmark_returns#超额收益
    bt_annual_return = float(
        bt_strategy_nav.iloc[-1]
        ** (BT_ANNUAL_TRADING_DAYS / bt_observations)
        - 1.0
    )#年化收益
    bt_annual_volatility = float(
        bt_strategy_returns.std(ddof=1) * np.sqrt(BT_ANNUAL_TRADING_DAYS)
    )#年化波动
    bt_strategy_excess_returns = (
        bt_strategy_returns - bt_risk_free_daily
    )#策略日收益减日无风险收益
    bt_sharpe = (
        float(
            bt_strategy_excess_returns.mean()
            / bt_strategy_excess_returns.std(ddof=1)
            * np.sqrt(BT_ANNUAL_TRADING_DAYS)
        )
        if bt_strategy_excess_returns.std(ddof=1) > 0
        else np.nan
    )#使用日无风险收益率的年化夏普
    bt_excess_annual_return = float(
        (bt_strategy_nav.iloc[-1] / bt_benchmark_nav.iloc[-1])
        ** (BT_ANNUAL_TRADING_DAYS / bt_observations)
        - 1.0
    )
    bt_active_volatility = float(
        bt_active_returns.std(ddof=1) * np.sqrt(BT_ANNUAL_TRADING_DAYS)
    )#超额收益的年化标准差
    bt_information_ratio = (
        float(
            bt_active_returns.mean()
            / bt_active_returns.std(ddof=1)
            * np.sqrt(BT_ANNUAL_TRADING_DAYS)
        )
        if bt_active_returns.std(ddof=1) > 0
        else np.nan
    )#年化信息比率
    bt_factor_turnover = bt_turnover_detail.loc[
        bt_turnover_detail["组合"].eq(bt_factor)
    ]#又单独拿出来
    bt_performance_rows.append({
        "因子": bt_factor,
        "累计收益率": float(bt_strategy_nav.iloc[-1] - 1.0),
        "年化收益率": bt_annual_return,
        "年化波动率": bt_annual_volatility,
        "夏普比率_日无风险收益率": bt_sharpe,
        "最大回撤": bt_max_drawdown(bt_strategy_nav),
        "相对沪深300超额年化收益率": bt_excess_annual_return,
        "信息比率": bt_information_ratio,
        "累计买入换手率": float(bt_factor_turnover["买入换手率"].sum()),
        "累计卖出换手率": float(bt_factor_turnover["卖出换手率"].sum()),
        "累计手续费": float(bt_factor_turnover["手续费"].sum()),
        "累计滑点成本": float(bt_factor_turnover["滑点成本"].sum()),
        "累计交易成本": float(bt_factor_turnover["总交易成本"].sum()),
        "期末净值": float(bt_strategy_nav.iloc[-1]),
    })

bt_performance = pd.DataFrame(bt_performance_rows).set_index("因子")
bt_performance.to_csv(BT_PERFORMANCE_PATH, encoding="utf-8-sig")

bt_percentage_columns = [
    "累计收益率",
    "年化收益率",
    "年化波动率",
    "最大回撤",
    "相对沪深300超额年化收益率",
    "累计买入换手率",
    "累计卖出换手率",
    "累计手续费",
    "累计滑点成本",
    "累计交易成本",
]
display(
    bt_performance.style.format({#决定展示的格式
        **{column: "{:.2%}" for column in bt_percentage_columns},
        "夏普比率_日无风险收益率": "{:.3f}",
        "信息比率": "{:.3f}",
        "期末净值": "{:.4f}",
    })
)

bt_plot_columns = [BT_BENCHMARK_NAME, *BT_FACTORS]#决定画图的列
bt_plot_labels = {
    BT_BENCHMARK_NAME: "沪深300指数",
    "EP": "EP增强",
    "BP": "BP增强",
    "SP": "SP增强",
}
bt_plot_colors = {
    BT_BENCHMARK_NAME: "#4B5563",
    "EP": "#2563EB",
    "BP": "#D97706",
    "SP": "#9D174D",
}
bt_ax = bt_nav_table[bt_plot_columns].rename(columns=bt_plot_labels).plot(#改下名字
    figsize=(12, 6),
    color=[bt_plot_colors[column] for column in bt_plot_columns],
    linewidth=1.8,
)
bt_ax.set_title("单因子指数增强组合与沪深300指数净值（2016年试跑）")
bt_ax.set_xlabel("日期")
bt_ax.set_ylabel("净值（初始=1）")
bt_ax.grid(axis="y", color="#D1D5DB", linewidth=0.7, alpha=0.7)
bt_ax.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.savefig(BT_NAV_PLOT_PATH, dpi=160, bbox_inches="tight")
plt.show()

bt_unknown = bt_monthly_industry_weights.loc[
    bt_monthly_industry_weights["行业一级代码"].eq("UNKNOWN")
]
bt_end_cash_limited = bt_end_pending_detail["待成交原因"].eq(
    "卖出受阻导致可用现金不足"
)
bt_end_direct_blocked = ~bt_end_cash_limited
bt_final_checks = pd.Series({
    "无风险收益率匹配交易日数": int(bt_risk_free_daily.notna().sum()),
    "无风险收益率缺失交易日数": int(bt_risk_free_daily.isna().sum()),
    "回测期日无风险收益率均值": float(bt_risk_free_daily.mean()),
    "月末信号数": len(bt_signal_calendar),
    "首个调仓日": bt_signal_calendar["调仓日"].min(),
    "最后调仓日": bt_signal_calendar["调仓日"].max(),
    "行业权重和偏离1最大值": float((bt_industry_weight_sums - 1.0).abs().max()),
    "行业缺失成分股记录数": int(
        bt_constituent_weights["行业一级代码"].eq("UNKNOWN").sum()
    ),
    "单月行业缺失权重最大值": float(bt_unknown["行业权重"].max()),
    "股票加现金权重和偏离1最大值": float(
        (bt_allocation_check["合计"] - 1.0).abs().max()
    ),
    "策略持仓收益缺失记录数": int(
        bt_daily_backtest.loc[
            bt_daily_backtest["组合"].isin(BT_FACTORS),
            "持仓收益缺失数"
        ].sum()
    ),

    "其中停牌导致收益缺失记录数": int(
        bt_daily_backtest.loc[
            bt_daily_backtest["组合"].isin(BT_FACTORS),
            "停牌导致收益缺失数"
        ].sum()
    ),

    "其中非停牌收益缺失记录数": int(
        bt_daily_backtest.loc[
            bt_daily_backtest["组合"].isin(BT_FACTORS),
            "非停牌收益缺失数"
        ].sum()
    ),
    "停牌股票日记录数": int(bt_tradeability["是否停牌"].sum()),
    "一字涨停股票日记录数": int(bt_tradeability["是否一字涨停"].sum()),
    "一字跌停股票日记录数": int(bt_tradeability["是否一字跌停"].sum()),
    "停牌受阻卖出次数": int(bt_turnover_detail["停牌受阻卖出数"].sum()),
    "一字跌停受阻卖出次数": int(
        bt_turnover_detail["一字跌停受阻卖出数"].sum()
    ),
    "期末未完成订单组合-股票项数": len(bt_end_pending_detail),
    "期末未完成订单独立股票数": int(
        bt_end_pending_detail[BT_CODE].nunique()
    ),
    "期末直接不可交易订单项数": int(bt_end_direct_blocked.sum()),
    "其中停牌卖出受阻订单项数": int((
        bt_end_pending_detail["待成交方向"].eq("卖出")
        & bt_end_pending_detail["是否停牌"]
    ).sum()),
    "其中停牌买入受阻订单项数": int((
        bt_end_pending_detail["待成交方向"].eq("买入")
        & bt_end_pending_detail["是否停牌"]
    ).sum()),
    "期末卖出受阻导致现金不足买入项数": int(
        bt_end_cash_limited.sum()
    ),
    "回测结束到期取消订单项数": len(bt_end_pending_detail),
    "回测结束后仍在市场等待成交订单项数": 0,
    "沪深300指数收益缺失交易日数": int(
        bt_hs300_returns.isna().sum()
    ),
    "期末清仓费": 0.0,
}, name="检查结果")#索引加结果
display(bt_final_checks.to_frame())

print(f"日无风险收益率：{BT_RISK_FREE_PATH}")
print(f"月度行业权重：{BT_INDUSTRY_WEIGHT_PATH}")
print(f"月度持仓：{BT_HOLDINGS_PATH}")
print(f"日度回测：{BT_DAILY_RESULT_PATH}")
print(f"可交易状态：{BT_TRADEABILITY_PATH}")
print(f"成交、换手与受阻明细：{BT_TURNOVER_PATH}")
print(f"期末到期取消订单明细：{BT_END_PENDING_PATH}")
print(f"绩效指标：{BT_PERFORMANCE_PATH}")
print(f"净值图：{BT_NAV_PLOT_PATH}")

,累计收益率,年化收益率,年化波动率,夏普比率_日无风险收益率,最大回撤,相对沪深300超额年化收益率,信息比率,累计买入换手率,累计卖出换手率,累计手续费,累计滑点成本,累计交易成本,期末净值
因子,,,,,,,,,,,,,
EP,25.55%,29.17%,22.27%,1.171,-10.27%,13.00%,1.875,251.28%,153.95%,1.84%,0.23%,2.07%,1.2555
BP,22.12%,25.21%,21.30%,1.068,-9.97%,9.53%,1.746,182.74%,84.77%,1.16%,0.15%,1.31%,1.2212
SP,22.39%,25.52%,20.34%,1.121,-9.33%,9.81%,1.937,198.83%,101.02%,1.32%,0.17%,1.49%,1.2239


C:\Temp\ipykernel_42532\4078128862.py:182: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,检查结果
无风险收益率匹配交易日数,224
无风险收益率缺失交易日数,0
回测期日无风险收益率均值,0.00008
月末信号数,11
首个调仓日,2016-02-01 00:00:00
最后调仓日,2016-12-01 00:00:00
行业权重和偏离1最大值,0.0
行业缺失成分股记录数,44
单月行业缺失权重最大值,0.00707
股票加现金权重和偏离1最大值,0.0


日无风险收益率：D:\因子分析\数据\RESSET_BDDRFRET_1.xlsx
月度行业权重：D:\因子分析\数据\回测结果_交易约束与滑点版\hs300_industry_weights_2016.csv
月度持仓：D:\因子分析\数据\回测结果_交易约束与滑点版\single_factor_monthly_holdings_2016.parquet
日度回测：D:\因子分析\数据\回测结果_交易约束与滑点版\single_factor_daily_backtest_2016.parquet
可交易状态：D:\因子分析\数据\回测结果_交易约束与滑点版\daily_tradeability_2016.parquet
成交、换手与受阻明细：D:\因子分析\数据\回测结果_交易约束与滑点版\single_factor_execution_detail_2016.parquet
期末到期取消订单明细：D:\因子分析\数据\回测结果_交易约束与滑点版\single_factor_end_pending_orders_2016.csv
绩效指标：D:\因子分析\数据\回测结果_交易约束与滑点版\single_factor_performance_2016.csv
净值图：D:\因子分析\数据\回测结果_交易约束与滑点版\single_factor_nav_2016.png
